# Sports Injury EM-Centered Report Analysis

This notebook is a report-oriented, step-by-step analysis for the project:
**ODE- and EM-Based Modeling of Sports Injury and Recovery Dynamics**.

The notebook is written so you can move block by block. Each major code cell is heavily commented,
and each major table or figure is followed by an interpretation cell that explains what it means and
whether it supports the modeling direction.


## Notebook Roadmap

This notebook follows the same logic as the report:

1. **Introduction and model setup**
2. **Data loading and variable identification**
3. **Preprocessing and construction of ODE inputs**
4. **Formal statistical justification for the preliminary ODE**
5. **Core descriptive and dynamic results**
6. **EM-centered latent recovery profile analysis**
7. **Cluster validation with additional variables**
8. **Conclusion / discussion**

The emphasis is now slightly different from the earlier notebook. The ODE remains important, but it is used
mainly as a **mechanistic first stage** that produces athlete-level dynamic summaries. The main downstream
analysis is the EM / Gaussian-mixture recovery-profile analysis.


## 1. Introduction

Training load is widely discussed as a driver of injury risk, but the empirical evidence is mixed and often
depends on how load is defined and how time is handled. Methodological critiques and reviews have argued that
many sports injury analyses rely on simple exposure summaries while overlooking the temporal structure of
adaptation and recovery [1, 2]. At the same time, fitness-fatigue and impulse-response models have long been
used to represent how training accumulates and dissipates over time, especially in performance monitoring
settings [3, 4]. More recent sports-injury reviews have also stressed that interpretable temporal mechanisms
remain underdeveloped in many applied injury models [2, 3].

The gap addressed by this project is therefore not just whether workload is associated with injury, but whether
athlete monitoring data contain a **temporally structured fatigue signal** that is better understood with a
dynamical model than with session-by-session summaries alone. A second gap is whether athletes appear to follow
one common recovery pattern or whether there is evidence for **latent recovery heterogeneity** that would justify
a later EM extension.


## 1A. Mathematical Model

The report is centered around three equations.

The latent fatigue state evolves according to

$$
\dot{x}(t) = \alpha u(t) - k x(t)
$$

where:

- $u(t)$ is the workload input,
- $\alpha$ is the accumulation rate,
- $k$ is the recovery rate.

The observed fatigue proxy satisfies

$$
y_i = x(t_i;\theta) + \epsilon_i, \qquad \epsilon_i \sim \mathcal{N}(0,\sigma^2)
$$

and a gradient-matching approximation motivates the diagnostic objective

$$
\min_{\alpha, k} \sum_i \left( \widehat{\dot{x}}(t_i) - \alpha u(t_i) + k \widehat{x}(t_i) \right)^2
$$

In this notebook, the ODE is treated as a **mechanistic first-stage model** rather than the final endpoint.
The first task is to check whether Equations (1)–(3) are statistically defensible. If they are, athlete-level
summaries derived from that preliminary model can be carried into an EM analysis for latent recovery profiles.


## 1B. How To Read The Code

This notebook is the narrative version of the project. The `src/athlete_recovery/` package in the repository contains the same core logic in reusable Python modules, but the notebook keeps the full analysis in a report-like order so the reasoning is visible cell by cell.

A good way to read it is:

1. follow the preprocessing cells to see how the panel and model variables are constructed;
2. read the ODE transition cells as a statistical justification step, not as a final model;
3. read the EM section as the main analysis, where athlete-level recovery summaries are turned into latent profiles;
4. use the later validation cells to judge whether those latent profiles actually mean anything outside the clustering variables.

The code comments are intentionally tied to the modeling decisions. They are there to explain *why* each block exists, not only what the syntax is doing.

In [ ]:
# Step 0: import the libraries used throughout the notebook.
#
# This notebook is intentionally written in a report-like order, but the code is
# still organized around four concrete jobs:
# 1. load and clean a longitudinal athlete panel;
# 2. test whether a simple ODE-style transition is statistically defensible;
# 3. turn that transition into athlete-level recovery features;
# 4. use EM / Gaussian mixtures to test for hidden recovery profiles.
#
# The imports reflect that structure. Pandas and NumPy handle panel data and
# feature construction, statsmodels handles the transition regressions, and
# scikit-learn handles cross-validation, scaling, PCA, and Gaussian mixtures.
# The plotting libraries are used to make report-quality figures rather than
# generic exploratory plots.
# Keeping this in one place makes the notebook easy to restart and rerun.
import os
from pathlib import Path
import warnings
import zipfile

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D
from IPython.display import Markdown, display
from scipy.stats import chi2_contingency, f_oneway, kruskal, kurtosis, skew

import statsmodels.api as sm
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import adjusted_rand_score, mean_absolute_error, mean_squared_error, r2_score, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

# Suppress clutter from third-party packages so the notebook remains readable.
warnings.filterwarnings("ignore")

# Set plotting defaults that work well both in the notebook and when exported
# into PNGs for the final report.
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "font.size": 10,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 8,
        "legend.title_fontsize": 8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.18,
        "grid.linewidth": 0.5,
    }
)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "final_report_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

INJURY_LABELS = {0: "Healthy", 1: "Low Risk", 2: "Injured"}
RANDOM_STATE = 42


def save_table(frame: pd.DataFrame, filename: str) -> Path:
    '''Save a table to the report output folder.'''
    path = OUTPUT_DIR / filename
    frame.to_csv(path, index=False)
    return path


def save_figure(fig: plt.Figure, filename: str) -> Path:
    '''Save a figure using report-friendly settings.'''
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight")
    return path


In [ ]:
# Step 1: load the local zipped dataset.
#
# The project is packaged so the notebook can run from the repository root with
# no manual file hunting. The code reads the first CSV inside the zip archive and
# keeps the raw frame unchanged so the preprocessing decisions are explicit later.
# That matters because the report needs to explain what came from the data as-is
# and what was constructed during cleaning and modeling.
# The project folder already contains the archive, so the notebook reads it directly.
zip_candidates = sorted(PROJECT_ROOT.glob("*.zip"))
if not zip_candidates:
    raise FileNotFoundError(f"No zip file found in {PROJECT_ROOT}")

zip_path = zip_candidates[0]
with zipfile.ZipFile(zip_path) as archive:
    csv_members = [name for name in archive.namelist() if name.lower().endswith(".csv")]
    if not csv_members:
        raise FileNotFoundError("The zip archive does not contain a CSV file.")
    csv_name = csv_members[0]
    with archive.open(csv_name) as handle:
        raw_df = pd.read_csv(handle)

print(f"ZIP file: {zip_path.name}")
print(f"CSV file: {csv_name}")
print(f"Shape: {raw_df.shape}")
print("\nColumns:")
print(raw_df.columns.tolist())

raw_df.head()


In [ ]:
# Step 2: explicitly map report concepts to dataset columns.
# This notebook checks that the variables needed for the ODE framing actually exist.
required_columns = {
    "athlete_id",
    "session_id",
    "injury_occurred",
    "training_load",
    "training_intensity",
    "training_duration",
    "fatigue_index",
    "recovery_score",
}
missing_required = sorted(required_columns - set(raw_df.columns))
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

variable_map = pd.DataFrame(
    {
        "role": [
            "athlete_id",
            "time_var",
            "injury_var",
            "u(t)",
            "y(t)",
            "alternate workload proxy",
        ],
        "dataset column": [
            "athlete_id",
            "session_id",
            "injury_occurred",
            "training_load",
            "fatigue_index",
            "training_intensity x training_duration",
        ],
        "note": [
            "panel identifier",
            "ordered within-athlete session index",
            "0=Healthy, 1=Low Risk, 2=Injured",
            "main workload input",
            "main observed fatigue proxy",
            "proxy check for workload construction",
        ],
    }
)

save_table(variable_map, "table_identified_variables.csv")
variable_map


In [ ]:
# Step 3: inspect missingness before any preprocessing.
# This is important because the Methods section needs to describe what data quality issues existed.
overall_missing_pct = 100 * raw_df.isna().sum().sum() / raw_df.size
missingness_table = (
    raw_df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_pct")
    .reset_index()
    .rename(columns={"index": "column"})
)

print(f"Overall missingness across the full table: {overall_missing_pct:.3f}%")
save_table(missingness_table, "table_missingness.csv")
missingness_table.head(10)


In [ ]:
# Step 4: clean and preprocess the dataset.
#
# This cell does most of the panel construction work, so it is worth reading with
# the model in mind.
#
# The choices here are tied directly to the later mathematics:
# - athlete_id defines the panel unit;
# - session_id defines the within-athlete time order;
# - training_load becomes u(t), the workload input;
# - fatigue_index becomes y(t), the observed fatigue proxy.
#
# Numeric imputation is done at the athlete level first because repeated measures
# from the same athlete are usually more informative than a single global median.
# After the imputation step, the code builds both centered and trailing rolling
# fatigue smoothers. The centered version is useful for descriptive plots, while
# the trailing version is the one used for transition validation because it does
# not use future information.
# We sort within athlete, impute missing values, and then construct
# the variables needed for the ODE-oriented analysis.
athlete_id = "athlete_id"
time_var = "session_id"
injury_var = "injury_occurred"

df = raw_df.copy().sort_values([athlete_id, time_var]).reset_index(drop=True)

# Impute numeric columns using athlete-level medians first.
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
numeric_impute_columns = [
    column for column in numeric_columns if column not in {athlete_id, injury_var, time_var}
]
for column in numeric_impute_columns:
    if df[column].isna().any():
        df[column] = df.groupby(athlete_id)[column].transform(lambda s: s.fillna(s.median()))
        df[column] = df[column].fillna(df[column].median())

# Impute categorical columns with the global mode.
categorical_columns = [column for column in df.columns if column not in numeric_columns]
for column in categorical_columns:
    if df[column].isna().any():
        mode = df[column].mode(dropna=True)
        if not mode.empty:
            df[column] = df[column].fillna(mode.iloc[0])

# Construct the main model variables.
df["u_t"] = df["training_load"]
df["u_intensity_x_duration"] = df["training_intensity"] * df["training_duration"]
df["y_t"] = df["fatigue_index"]

# Construct smoothed and rolling features used for dynamic diagnostics.
df["y_t_smooth_centered"] = (
    df.groupby(athlete_id)["y_t"]
    .transform(lambda s: s.rolling(window=3, min_periods=1, center=True).mean())
)
df["y_t_smooth_past"] = (
    df.groupby(athlete_id)["y_t"]
    .transform(lambda s: s.rolling(window=3, min_periods=1).mean())
)
df["y_t_smooth"] = df["y_t_smooth_centered"]
df["u_ma_3"] = (
    df.groupby(athlete_id)["u_t"]
    .transform(lambda s: s.rolling(window=3, min_periods=1).mean())
)

# Approximate the derivative used in the gradient-matching check.
df["delta_t"] = df.groupby(athlete_id)[time_var].diff().astype(float).replace(0, np.nan)
df["dy_dt"] = df.groupby(athlete_id)["y_t_smooth"].diff() / df["delta_t"]

# Build lagged injury indicators for onset and next-session analyses.
df["prev_injury"] = df.groupby(athlete_id)[injury_var].shift(1)
df["next_injury"] = df.groupby(athlete_id)[injury_var].shift(-1)
df["next_injured"] = (df["next_injury"] == 2).astype(float)
df["injury_onset"] = df[injury_var].eq(2) & df["prev_injury"].fillna(0).lt(2)

# Summarize the cleaned panel.
panel_counts = df.groupby(athlete_id).size()
session_gaps = df.groupby(athlete_id)[time_var].diff().dropna()

panel_overview = pd.DataFrame(
    {
        "metric": [
            "observations",
            "athletes",
            "sessions per athlete (mean)",
            "sessions per athlete (min-max)",
            "session index span",
            "overall missingness (%) before imputation",
            "largest column missingness (%) before imputation",
            "non-unit session gaps detected",
        ],
        "value": [
            f"{len(df):,}",
            f"{df[athlete_id].nunique()}",
            f"{panel_counts.mean():.2f}",
            f"{panel_counts.min()}-{panel_counts.max()}",
            f"{df[time_var].min()}-{df[time_var].max()}",
            f"{overall_missing_pct:.3f}",
            f"{missingness_table.loc[0, 'missing_pct']:.3f} ({missingness_table.loc[0, 'column']})",
            "No" if session_gaps.min() == 1.0 and session_gaps.max() == 1.0 else "Yes",
        ],
    }
)

save_table(panel_overview, "table_panel_overview.csv")
panel_overview


In [ ]:
# Step 5: write down the practical model choices in one place.
# This makes it easy to lift the wording into the Methods section later.
model_choices = pd.DataFrame(
    {
        "component": [
            "u(t)",
            "y(t)",
            "time index",
            "descriptive smoothing rule",
            "transition-validation smoothing rule",
            "injury-onset rule",
        ],
        "definition": [
            "training_load",
            "fatigue_index",
            "session_id within athlete",
            "3-session centered rolling mean for y(t)",
            "3-session trailing rolling mean for y(t), used to avoid look-ahead leakage",
            "injury_occurred == 2 following a previous session coded < 2",
        ],
    }
)

save_table(model_choices, "table_model_choices.csv")
model_choices


## 2A. Why This ODE Is Not Arbitrary

A model like

$$
\dot{x}(t) = \alpha u(t) - k x(t)
$$

is not something the dataset can "prove" from first principles. It is a **mechanistic hypothesis**
motivated by first-order impulse-response and fitness-fatigue thinking: workload pushes fatigue upward,
while recovery pulls it back down.

What the data *can* do is test whether this hypothesis is statistically defensible. Because `session_id`
increases by exactly one within each athlete, a unit-step Euler approximation gives

$$
y_{a,t+1} \approx c + \rho y_{a,t} + \alpha u_{a,t}, \qquad \rho = 1-k
$$

so the ODE has clear empirical implications:

- $\alpha > 0$: higher workload should increase next-session fatigue burden.
- $0 < \rho < 1$, equivalently $k > 0$: fatigue should persist, but decay rather than explode.
- The ODE transition model should beat simpler baselines such as intercept-only, workload-only, or persistence-only models.
- The coefficient signs should be broadly stable across athletes, not driven by a handful of outliers.

Because `fatigue_index` is an observed proxy rather than the latent state itself, the notebook checks both
the **raw** series and a **past-only 3-session rolling mean**. The past-only smoother is important for
transition validation because it reduces measurement noise **without using future information**.


In [ ]:
# Step 5A: build transition data and fit the discrete-time versions of the ODE.
#
# This is the first real statistical test of the ODE idea.
#
# The continuous model says
#     dx/dt = alpha * u(t) - k * x(t).
# At the session level, that implies a one-step transition in which next-session
# fatigue should depend positively on current workload and positively, but not too
# strongly, on current fatigue. If the workload coefficient comes out negative or
# the persistence term is implausible, the simple ODE is not a good first-stage
# model.
#
# The cell fits three versions on purpose:
# - raw observed fatigue,
# - trailing-smoothed fatigue,
# - and a within-athlete demeaned version.
#
# The raw fit is a strict falsification check. The smoothed fit asks whether the
# latent-state story works once measurement noise is reduced. The demeaned fit asks
# whether the result is still present after removing athlete-specific levels.
# This is the first formal statistical justification step.

# Create one-step-ahead targets for both the raw fatigue proxy and the past-only
# smoothed version used for transition validation.
transition_df = df[
    [
        athlete_id,
        time_var,
        injury_var,
        "injury_onset",
        "u_t",
        "y_t",
        "y_t_smooth_past",
        "recovery_score",
        "sleep_quality",
        "stress_level",
    ]
].copy()
transition_df["y_next_raw"] = transition_df.groupby(athlete_id)["y_t"].shift(-1)
transition_df["y_next_smooth_past"] = transition_df.groupby(athlete_id)["y_t_smooth_past"].shift(-1)
transition_df["dy_next_smooth_past"] = transition_df["y_next_smooth_past"] - transition_df["y_t_smooth_past"]


def fit_clustered_transition(frame: pd.DataFrame, response: str, predictors: list[str], cluster_col: str):
    '''Fit OLS with athlete-clustered standard errors.'''
    model_frame = frame[[cluster_col, response] + predictors].dropna().copy()
    X = sm.add_constant(model_frame[predictors], has_constant="add")
    result = sm.OLS(model_frame[response], X).fit(
        cov_type="cluster",
        cov_kwds={"groups": model_frame[cluster_col]},
    )
    return result, model_frame


def fit_demeaned_transition(frame: pd.DataFrame, response: str, predictors: list[str], group_col: str):
    '''Remove athlete means first so the regression uses only within-athlete variation.'''
    model_frame = frame[[group_col, response] + predictors].dropna().copy()
    demeaned_map = {}
    for column in [response] + predictors:
        demeaned_column = f"{column}_dm"
        model_frame[demeaned_column] = model_frame[column] - model_frame.groupby(group_col)[column].transform("mean")
        demeaned_map[column] = demeaned_column
    X = model_frame[[demeaned_map[column] for column in predictors]]
    result = sm.OLS(model_frame[demeaned_map[response]], X).fit(
        cov_type="cluster",
        cov_kwds={"groups": model_frame[group_col]},
    )
    return result, model_frame, demeaned_map


# 1. Fit the raw one-step transition. This is intentionally strict:
# it asks whether the observed fatigue proxy can directly act like the latent state.
raw_transition_result, raw_transition_data = fit_clustered_transition(
    transition_df,
    response="y_next_raw",
    predictors=["y_t", "u_t"],
    cluster_col=athlete_id,
)

# 2. Fit the same transition after replacing y(t) with a past-only smoothed proxy.
# This is usually the more appropriate test when the observation model includes noise.
smoothed_transition_result, smoothed_transition_data = fit_clustered_transition(
    transition_df,
    response="y_next_smooth_past",
    predictors=["y_t_smooth_past", "u_t"],
    cluster_col=athlete_id,
)

# 3. Repeat the smoothed transition with athlete demeaning to show that the signal
# persists after removing athlete-specific levels.
within_transition_result, within_transition_data, demeaned_map = fit_demeaned_transition(
    transition_df,
    response="y_next_smooth_past",
    predictors=["y_t_smooth_past", "u_t"],
    group_col=athlete_id,
)

# 4. Fit the derivative-style regression implied by the Euler step:
# delta y_t ≈ c + alpha * u_t - k * y_t.
derivative_transition_result, derivative_transition_data = fit_clustered_transition(
    transition_df,
    response="dy_next_smooth_past",
    predictors=["u_t", "y_t_smooth_past"],
    cluster_col=athlete_id,
)


transition_summary_table = pd.DataFrame(
    [
        {
            "specification": "Raw observed transition",
            "response": "y_(t+1)",
            "alpha_hat": raw_transition_result.params["u_t"],
            "alpha_se": raw_transition_result.bse["u_t"],
            "alpha_pvalue": raw_transition_result.pvalues["u_t"],
            "rho_hat": raw_transition_result.params["y_t"],
            "rho_se": raw_transition_result.bse["y_t"],
            "rho_pvalue": raw_transition_result.pvalues["y_t"],
            "implied_k": 1 - raw_transition_result.params["y_t"],
            "r_squared": raw_transition_result.rsquared,
            "adj_r_squared": raw_transition_result.rsquared_adj,
            "n_obs": len(raw_transition_data),
        },
        {
            "specification": "Past-smoothed transition",
            "response": "y~_(t+1)",
            "alpha_hat": smoothed_transition_result.params["u_t"],
            "alpha_se": smoothed_transition_result.bse["u_t"],
            "alpha_pvalue": smoothed_transition_result.pvalues["u_t"],
            "rho_hat": smoothed_transition_result.params["y_t_smooth_past"],
            "rho_se": smoothed_transition_result.bse["y_t_smooth_past"],
            "rho_pvalue": smoothed_transition_result.pvalues["y_t_smooth_past"],
            "implied_k": 1 - smoothed_transition_result.params["y_t_smooth_past"],
            "r_squared": smoothed_transition_result.rsquared,
            "adj_r_squared": smoothed_transition_result.rsquared_adj,
            "n_obs": len(smoothed_transition_data),
        },
        {
            "specification": "Past-smoothed transition, within-athlete",
            "response": "y~_(t+1) demeaned",
            "alpha_hat": within_transition_result.params[demeaned_map["u_t"]],
            "alpha_se": within_transition_result.bse[demeaned_map["u_t"]],
            "alpha_pvalue": within_transition_result.pvalues[demeaned_map["u_t"]],
            "rho_hat": within_transition_result.params[demeaned_map["y_t_smooth_past"]],
            "rho_se": within_transition_result.bse[demeaned_map["y_t_smooth_past"]],
            "rho_pvalue": within_transition_result.pvalues[demeaned_map["y_t_smooth_past"]],
            "implied_k": 1 - within_transition_result.params[demeaned_map["y_t_smooth_past"]],
            "r_squared": within_transition_result.rsquared,
            "adj_r_squared": within_transition_result.rsquared_adj,
            "n_obs": len(within_transition_data),
        },
    ]
)

derivative_summary_table = pd.DataFrame(
    [
        {
            "specification": "Past-smoothed derivative regression",
            "response": "delta y~_t",
            "alpha_hat": derivative_transition_result.params["u_t"],
            "alpha_se": derivative_transition_result.bse["u_t"],
            "alpha_pvalue": derivative_transition_result.pvalues["u_t"],
            "y_coefficient_hat": derivative_transition_result.params["y_t_smooth_past"],
            "y_coefficient_se": derivative_transition_result.bse["y_t_smooth_past"],
            "y_coefficient_pvalue": derivative_transition_result.pvalues["y_t_smooth_past"],
            "implied_k": -derivative_transition_result.params["y_t_smooth_past"],
            "r_squared": derivative_transition_result.rsquared,
            "adj_r_squared": derivative_transition_result.rsquared_adj,
            "n_obs": len(derivative_transition_data),
        }
    ]
)

save_table(transition_summary_table, "table_transition_model_justification.csv")
save_table(derivative_summary_table, "table_derivative_model_justification.csv")

display(Markdown("### Transition-model justification table"))
display(transition_summary_table)
display(Markdown("### Derivative-model justification table"))
derivative_summary_table


In [ ]:
# Step 5B: compare the ODE transition model against simpler baselines,
# then check whether the estimated parameters are stable across athletes.
#
# A transition model can look good in-sample for the wrong reason, so this cell
# asks three harder questions.
#
# First, does the ODE-style one-step model beat simpler alternatives when whole
# athletes are held out in grouped cross-validation?
#
# Second, if the pooled model works, does it work because most athletes show the
# same sign pattern, or only because a few athletes dominate the fit?
#
# Third, do recent workload lags still line up with the fatigue story in a way
# that looks like short-term accumulation rather than random correlation?
#
# Those checks matter because the ODE is only useful here if it gives a stable and
# interpretable athlete-level summary for the later EM step.
# then check whether the estimated parameters are stable across athletes.

def grouped_cv_table(
    frame: pd.DataFrame,
    response: str,
    feature_map: dict[str, list[str]],
    group_col: str,
    n_splits: int = 5,
) -> pd.DataFrame:
    rows = []
    splitter = GroupKFold(n_splits=n_splits)
    groups = frame[group_col]
    for label, predictors in feature_map.items():
        y_true = []
        y_pred = []
        for train_idx, test_idx in splitter.split(frame, groups=groups):
            train = frame.iloc[train_idx]
            test = frame.iloc[test_idx]
            if predictors:
                model = LinearRegression().fit(train[predictors], train[response])
                predictions = model.predict(test[predictors])
            else:
                predictions = np.repeat(train[response].mean(), len(test))
            y_true.extend(test[response].tolist())
            y_pred.extend(predictions.tolist())
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        rows.append(
            {
                "model": label,
                "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
                "mae": mean_absolute_error(y_true, y_pred),
                "r_squared": r2_score(y_true, y_pred),
            }
        )
    return pd.DataFrame(rows)


cv_frame = transition_df[
    [athlete_id, "u_t", "y_t_smooth_past", "y_next_smooth_past", "recovery_score", "sleep_quality", "stress_level"]
].dropna().copy()

cv_feature_map = {
    "Intercept only": [],
    "Workload only": ["u_t"],
    "Persistence only": ["y_t_smooth_past"],
    "ODE transition": ["y_t_smooth_past", "u_t"],
    "Contextual benchmark": ["y_t_smooth_past", "u_t", "recovery_score", "sleep_quality", "stress_level"],
}
cv_model_comparison = grouped_cv_table(
    cv_frame,
    response="y_next_smooth_past",
    feature_map=cv_feature_map,
    group_col=athlete_id,
)

persistence_row = cv_model_comparison.loc[cv_model_comparison["model"] == "Persistence only"].iloc[0]
cv_model_comparison["rmse_gain_vs_persistence"] = persistence_row["rmse"] - cv_model_comparison["rmse"]
cv_model_comparison["r2_gain_vs_persistence"] = cv_model_comparison["r_squared"] - persistence_row["r_squared"]

# Fit the smoothed transition model separately within each athlete.
# This checks whether the estimated accumulation and recovery signs are stable.
athlete_rows = []
for athlete, athlete_frame in smoothed_transition_data.groupby(athlete_id):
    if len(athlete_frame) < 20:
        continue
    X = sm.add_constant(athlete_frame[["y_t_smooth_past", "u_t"]], has_constant="add")
    result = sm.OLS(athlete_frame["y_next_smooth_past"], X).fit()
    full_athlete_frame = df[df[athlete_id] == athlete]
    k_hat = 1 - result.params["y_t_smooth_past"]
    athlete_rows.append(
        {
            athlete_id: athlete,
            "n_obs": len(athlete_frame),
            "alpha_hat": result.params["u_t"],
            "rho_hat": result.params["y_t_smooth_past"],
            "k_hat": k_hat,
            "half_life_sessions": np.log(2) / k_hat if k_hat > 0 else np.nan,
            "r_squared": result.rsquared,
            "injury_rate_any": (full_athlete_frame[injury_var] > 0).mean(),
            "injury_rate_injured": (full_athlete_frame[injury_var] == 2).mean(),
            "mean_recovery_score": full_athlete_frame["recovery_score"].mean(),
            "mean_sleep_quality": full_athlete_frame["sleep_quality"].mean(),
            "mean_workload": full_athlete_frame["u_t"].mean(),
            "sport_type": full_athlete_frame["sport_type"].iloc[0],
            "gender": full_athlete_frame["gender"].iloc[0],
        }
    )
athlete_transition_detail = pd.DataFrame(athlete_rows)
athlete_transition_summary = pd.DataFrame(
    {
        "metric": [
            "athletes fit",
            "median alpha_hat",
            "median rho_hat",
            "median k_hat",
            "median half-life (sessions)",
            "median athlete R^2",
            "share alpha_hat > 0",
            "share 0 < rho_hat < 1",
            "share k_hat > 0",
        ],
        "value": [
            int(len(athlete_transition_detail)),
            athlete_transition_detail["alpha_hat"].median(),
            athlete_transition_detail["rho_hat"].median(),
            athlete_transition_detail["k_hat"].median(),
            athlete_transition_detail["half_life_sessions"].median(),
            athlete_transition_detail["r_squared"].median(),
            (athlete_transition_detail["alpha_hat"] > 0).mean(),
            ((athlete_transition_detail["rho_hat"] > 0) & (athlete_transition_detail["rho_hat"] < 1)).mean(),
            (athlete_transition_detail["k_hat"] > 0).mean(),
        ],
    }
)

# Add a lag-based check: if the ODE idea is reasonable, recent workload history should
# contribute positively to current fatigue burden, with strongest effects at short lags.
lag_df = df[[athlete_id, "u_t", "y_t_smooth_past"]].copy()
lag_columns = []
for lag in range(6):
    column = f"u_lag_{lag}"
    lag_df[column] = lag_df.groupby(athlete_id)["u_t"].shift(lag)
    lag_columns.append(column)
lag_model_df = lag_df[[athlete_id, "y_t_smooth_past"] + lag_columns].dropna().copy()
lag_model_df["y_t_smooth_past_dm"] = (
    lag_model_df["y_t_smooth_past"] - lag_model_df.groupby(athlete_id)["y_t_smooth_past"].transform("mean")
)
for column in lag_columns:
    lag_model_df[f"{column}_dm"] = lag_model_df[column] - lag_model_df.groupby(athlete_id)[column].transform("mean")

lag_predictors = [f"{column}_dm" for column in lag_columns]
lag_X = sm.add_constant(lag_model_df[lag_predictors], has_constant="add")
lag_result = sm.OLS(lag_model_df["y_t_smooth_past_dm"], lag_X).fit(
    cov_type="cluster",
    cov_kwds={"groups": lag_model_df[athlete_id]},
)
lag_workload_table = pd.DataFrame(
    [
        {
            "lag": lag,
            "coefficient": lag_result.params[f"u_lag_{lag}_dm"],
            "std_error": lag_result.bse[f"u_lag_{lag}_dm"],
            "pvalue": lag_result.pvalues[f"u_lag_{lag}_dm"],
        }
        for lag in range(6)
    ]
)

save_table(cv_model_comparison, "table_cv_model_comparison.csv")
save_table(athlete_transition_detail, "table_athlete_transition_detail.csv")
save_table(athlete_transition_summary, "table_athlete_transition_summary.csv")
save_table(lag_workload_table, "table_lagged_workload_coefficients.csv")

display(Markdown("### Grouped cross-validation model comparison"))
display(cv_model_comparison)
display(Markdown("### Athlete-level parameter stability summary"))
display(athlete_transition_summary)
display(Markdown("### Lagged workload check"))
lag_workload_table


In [ ]:
# Step 5C: create a compact visual summary of the model-justification results.
cv_plot_df = cv_model_comparison.copy()
cv_plot_df["model_short"] = [
    "Intercept",
    "Workload",
    "Persistence",
    "ODE",
    "Contextual",
]

fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.2), gridspec_kw={"width_ratios": [1.05, 1]})

sns.barplot(
    data=cv_plot_df,
    x="model_short",
    y="rmse",
    hue="model_short",
    palette=["#ced4da", "#74c69d", "#457b9d", "#e76f51", "#6d597a"],
    dodge=False,
    legend=False,
    edgecolor="none",
    ax=axes[0],
)
axes[0].set_title("Grouped CV: next-session fatigue prediction", pad=6)
axes[0].set_xlabel("")
axes[0].set_ylabel("RMSE")
axes[0].tick_params(axis="x", rotation=15)
for patch, value in zip(axes[0].patches, cv_plot_df["rmse"]):
    axes[0].annotate(
        f"{value:.2f}",
        (patch.get_x() + patch.get_width() / 2, patch.get_height()),
        ha="center",
        va="bottom",
        fontsize=8,
        xytext=(0, 2),
        textcoords="offset points",
    )

scatter = axes[1].scatter(
    athlete_transition_detail["alpha_hat"],
    athlete_transition_detail["k_hat"],
    c=athlete_transition_detail["r_squared"],
    cmap="viridis",
    s=26,
    alpha=0.85,
    linewidths=0,
)
axes[1].axhline(0, color="black", linestyle="--", linewidth=0.8)
axes[1].axvline(0, color="black", linestyle="--", linewidth=0.8)
axes[1].set_title("Athlete-specific ODE parameters", pad=6)
axes[1].set_xlabel("alpha_hat")
axes[1].set_ylabel("k_hat")
axes[1].text(
    0.03,
    0.97,
    f"{100 * (athlete_transition_detail['alpha_hat'] > 0).mean():.1f}% alpha_hat > 0\n"
    f"{100 * (athlete_transition_detail['k_hat'] > 0).mean():.1f}% k_hat > 0",
    transform=axes[1].transAxes,
    ha="left",
    va="top",
    fontsize=8,
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.85},
)
cbar = fig.colorbar(scatter, ax=axes[1], fraction=0.046, pad=0.02)
cbar.set_label("Athlete R^2")

fig.tight_layout(pad=0.5, w_pad=1.0)
save_figure(fig, "figure_0_model_justification.png")
plt.show()


In [ ]:
# Step 5D: interpret the formal justification tests directly and honestly.
persistence_cv = cv_model_comparison.loc[cv_model_comparison["model"] == "Persistence only"].iloc[0]
ode_cv = cv_model_comparison.loc[cv_model_comparison["model"] == "ODE transition"].iloc[0]
contextual_cv = cv_model_comparison.loc[cv_model_comparison["model"] == "Contextual benchmark"].iloc[0]

smoothed_alpha_p = smoothed_transition_result.pvalues["u_t"]
smoothed_rho_p = smoothed_transition_result.pvalues["y_t_smooth_past"]
derivative_y_p = derivative_transition_result.pvalues["y_t_smooth_past"]

smoothed_alpha_p_str = "<0.001" if smoothed_alpha_p < 0.001 else f"{smoothed_alpha_p:.3f}"
smoothed_rho_p_str = "<0.001" if smoothed_rho_p < 0.001 else f"{smoothed_rho_p:.3f}"
derivative_y_p_str = "<0.001" if derivative_y_p < 0.001 else f"{derivative_y_p:.3f}"

display(
    Markdown(
        f'''
**Evaluation of the formal ODE justification tests**

- The raw one-step transition is **not** convincing on its own: it gives a workload coefficient of
  **{raw_transition_result.params["u_t"]:.4f}**, which has the wrong sign for a fatigue-accumulation model.
  This is useful evidence rather than bad news, because it shows the observed `fatigue_index` is too noisy
  to be treated as the latent state directly.
- After replacing raw `y(t)` with a **past-only smoothed proxy**, the transition fit becomes mechanically sensible:
  $\hat{{\alpha}} = {smoothed_transition_result.params["u_t"]:.4f}$ (p {smoothed_alpha_p_str}),
  $\hat{{\rho}} = {smoothed_transition_result.params["y_t_smooth_past"]:.3f}$ (p {smoothed_rho_p_str}),
  implying $\hat{{k}} = {1 - smoothed_transition_result.params["y_t_smooth_past"]:.3f}$.
- The within-athlete demeaned fit keeps the same signs:
  $\hat{{\alpha}} = {within_transition_result.params[demeaned_map["u_t"]]:.4f}$ and
  $\hat{{k}} = {1 - within_transition_result.params[demeaned_map["y_t_smooth_past"]]:.3f}$.
  That matters because it means the ODE signal is not only a between-athlete difference.
- Grouped cross-validation shows that the ODE transition improves on a persistence-only baseline:
  RMSE drops from **{persistence_cv["rmse"]:.3f}** to **{ode_cv["rmse"]:.3f}**, and out-of-athlete
  $R^2$ rises from **{persistence_cv["r_squared"]:.3f}** to **{ode_cv["r_squared"]:.3f}**.
- A richer contextual benchmark still does a bit better (**RMSE {contextual_cv["rmse"]:.3f}**, $R^2$ **{contextual_cv["r_squared"]:.3f}**),
  so the ODE should be described as a **parsimonious mechanistic model**, not the unique best predictive model.
- The athlete-specific fits are stable: **{100 * (athlete_transition_detail["alpha_hat"] > 0).mean():.1f}%**
  of athletes have $\hat{{\alpha}} > 0$, and **{100 * (athlete_transition_detail["k_hat"] > 0).mean():.1f}%**
  have $\hat{{k}} > 0$.
- The derivative-style regression also recovers the expected recovery sign on the smoothed series:
  the coefficient on $y(t)$ is **{derivative_transition_result.params["y_t_smooth_past"]:.3f}** (p {derivative_y_p_str}),
  so the implied recovery rate remains positive.
- Recent workload lags remain positive through lag 5, which is consistent with accumulation rather than an isolated single-session effect.

**Are we on the right track?**

Yes, with the right level of rigor. The notebook can now justify the ODE statistically:
it is a reasonable first-order approximation once fatigue is treated as a noisy observation of a latent state.
The honest caveat is that this is a **defensible modeling choice**, not a proof that the true biology is exactly linear.
        '''
    )
)


## 2B. ODE Adequacy and Sensitivity Checks

The previous section asked whether the first-order ODE is *defensible at all*. This section goes further and
asks whether the specific choices we made are fragile.

The key diagnostics below examine:

- **Smoothing sensitivity:** does the sign pattern depend entirely on one arbitrary rolling window?
- **Workload-definition sensitivity:** do we get the same qualitative answer if workload is defined differently?
- **Model-shape sensitivity:** do simple nonlinear terms materially outperform the linear ODE transition?
- **Residual behavior:** does the fitted model leave obvious structure behind, especially around injury?
- **Subgroup stability:** do the ODE parameters look similar across sports and genders?

This is where the notebook moves from "the model seems plausible" to "the model survives several reasonable stress tests."


In [ ]:
# Step 5E: run sensitivity checks and deeper adequacy diagnostics for the ODE.
#
# Passing one transition regression is not enough. This cell checks whether the
# main ODE story survives reasonable changes in preprocessing and specification.
# In particular, it asks whether the sign pattern depends too heavily on one
# smoothing choice or one workload definition. If the answer were yes, the later
# EM stage would be built on unstable features and would be much harder to defend.

# 1. Smoothing sensitivity: repeat the smoothed one-step transition over several
# trailing rolling-window widths. This checks whether the ODE signs are fragile.
smoothing_rows = []
for window in [1, 2, 3, 5, 7]:
    y_column = f"y_t_smooth_past_w{window}"
    y_next_column = f"{y_column}_next"
    tmp = df[[athlete_id, "u_t", "y_t"]].copy()
    tmp[y_column] = tmp.groupby(athlete_id)["y_t"].transform(lambda s: s.rolling(window, min_periods=1).mean())
    tmp[y_next_column] = tmp.groupby(athlete_id)[y_column].shift(-1)
    model_frame = tmp[[athlete_id, "u_t", y_column, y_next_column]].dropna().copy()
    X = sm.add_constant(model_frame[[y_column, "u_t"]], has_constant="add")
    result = sm.OLS(model_frame[y_next_column], X).fit(
        cov_type="cluster",
        cov_kwds={"groups": model_frame[athlete_id]},
    )
    smoothing_rows.append(
        {
            "smoothing_window": window,
            "alpha_hat": result.params["u_t"],
            "rho_hat": result.params[y_column],
            "k_hat": 1 - result.params[y_column],
            "r_squared": result.rsquared,
            "adj_r_squared": result.rsquared_adj,
        }
    )
smoothing_sensitivity_table = pd.DataFrame(smoothing_rows)

# 2. Workload-definition sensitivity: compare the main workload definition to the
# alternative proxy intensity x duration after standardizing each predictor.
workload_rows = []
for workload_column, workload_label in [
    ("u_t", "training_load"),
    ("u_intensity_x_duration", "training_intensity x training_duration"),
]:
    workload_frame = df[[athlete_id, "y_t_smooth_past", workload_column]].copy()
    workload_frame["y_next"] = workload_frame.groupby(athlete_id)["y_t_smooth_past"].shift(-1)
    workload_frame = workload_frame.dropna().copy()
    standardized_column = f"{workload_column}_z"
    workload_frame[standardized_column] = (
        workload_frame[workload_column] - workload_frame[workload_column].mean()
    ) / workload_frame[workload_column].std()
    X = sm.add_constant(workload_frame[["y_t_smooth_past", standardized_column]], has_constant="add")
    result = sm.OLS(workload_frame["y_next"], X).fit(
        cov_type="cluster",
        cov_kwds={"groups": workload_frame[athlete_id]},
    )
    workload_rows.append(
        {
            "workload_definition": workload_label,
            "standardized_alpha_hat": result.params[standardized_column],
            "rho_hat": result.params["y_t_smooth_past"],
            "k_hat": 1 - result.params["y_t_smooth_past"],
            "r_squared": result.rsquared,
            "adj_r_squared": result.rsquared_adj,
        }
    )
workload_sensitivity_table = pd.DataFrame(workload_rows)

# 3. Model-shape sensitivity: compare the simple ODE transition to modest nonlinear
# extensions. If they barely help, the linear ODE is a sensible parsimonious choice.
model_shape_frame = cv_frame.copy()
model_shape_frame["u_sq"] = model_shape_frame["u_t"] ** 2
model_shape_frame["y_sq"] = model_shape_frame["y_t_smooth_past"] ** 2
model_shape_frame["u_y_interaction"] = model_shape_frame["u_t"] * model_shape_frame["y_t_smooth_past"]

model_shape_map = {
    "Persistence only": ["y_t_smooth_past"],
    "ODE transition": ["y_t_smooth_past", "u_t"],
    "ODE + quadratic workload": ["y_t_smooth_past", "u_t", "u_sq"],
    "ODE + interaction": ["y_t_smooth_past", "u_t", "u_y_interaction"],
    "ODE + quadratic y and u": ["y_t_smooth_past", "u_t", "u_sq", "y_sq"],
    "ODE + quadratic + interaction": ["y_t_smooth_past", "u_t", "u_sq", "y_sq", "u_y_interaction"],
    "Contextual benchmark": ["y_t_smooth_past", "u_t", "recovery_score", "sleep_quality", "stress_level"],
}
model_shape_comparison = grouped_cv_table(
    model_shape_frame,
    response="y_next_smooth_past",
    feature_map=model_shape_map,
    group_col=athlete_id,
)
ode_shape_row = model_shape_comparison.loc[model_shape_comparison["model"] == "ODE transition"].iloc[0]
model_shape_comparison["rmse_gain_vs_ode"] = ode_shape_row["rmse"] - model_shape_comparison["rmse"]
model_shape_comparison["r2_gain_vs_ode"] = model_shape_comparison["r_squared"] - ode_shape_row["r_squared"]

# 4. Residual diagnostics: examine calibration, remaining serial structure,
# and whether residuals are larger during injured sessions.
residual_diagnostic_df = smoothed_transition_data.copy()
residual_diagnostic_df["fitted"] = smoothed_transition_result.predict(
    sm.add_constant(residual_diagnostic_df[["y_t_smooth_past", "u_t"]], has_constant="add")
)
residual_diagnostic_df["residual"] = residual_diagnostic_df["y_next_smooth_past"] - residual_diagnostic_df["fitted"]
residual_diagnostic_df["abs_residual"] = residual_diagnostic_df["residual"].abs()
residual_diagnostic_df[injury_var] = df.loc[residual_diagnostic_df.index, injury_var].values
residual_diagnostic_df["sport_type"] = df.loc[residual_diagnostic_df.index, "sport_type"].values
residual_diagnostic_df["gender"] = df.loc[residual_diagnostic_df.index, "gender"].values

athlete_residual_lag1 = []
for _, athlete_frame in residual_diagnostic_df.groupby(athlete_id):
    lag_corr = athlete_frame["residual"].corr(athlete_frame["residual"].shift(1))
    if pd.notna(lag_corr):
        athlete_residual_lag1.append(lag_corr)
athlete_residual_lag1 = pd.Series(athlete_residual_lag1)

residual_summary_table = pd.DataFrame(
    {
        "metric": [
            "residual mean",
            "residual standard deviation",
            "residual skewness",
            "residual excess kurtosis",
            "median athlete lag-1 residual correlation",
            "mean athlete lag-1 residual correlation",
        ],
        "value": [
            residual_diagnostic_df["residual"].mean(),
            residual_diagnostic_df["residual"].std(),
            skew(residual_diagnostic_df["residual"]),
            kurtosis(residual_diagnostic_df["residual"], fisher=True),
            athlete_residual_lag1.median(),
            athlete_residual_lag1.mean(),
        ],
    }
)

residual_by_injury_table = (
    residual_diagnostic_df.groupby(injury_var)
    .agg(
        mean_residual=("residual", "mean"),
        residual_sd=("residual", "std"),
        mean_abs_residual=("abs_residual", "mean"),
        sessions=("residual", "size"),
    )
    .reset_index()
)
residual_by_injury_table["injury_label"] = residual_by_injury_table[injury_var].map(INJURY_LABELS)
residual_by_injury_table = residual_by_injury_table[
    ["injury_label", "mean_residual", "residual_sd", "mean_abs_residual", "sessions"]
]

calibration_table = residual_diagnostic_df[["fitted", "y_next_smooth_past"]].copy()
calibration_table["prediction_decile"] = pd.qcut(
    calibration_table["fitted"],
    10,
    labels=[f"D{i}" for i in range(1, 11)],
)
calibration_table = (
    calibration_table.groupby("prediction_decile", observed=False)
    .agg(
        mean_predicted=("fitted", "mean"),
        mean_observed=("y_next_smooth_past", "mean"),
        sessions=("fitted", "size"),
    )
    .reset_index()
)

# 5. Subgroup stability: fit the same ODE transition within sports and genders.
subgroup_rows = []
subgroup_specs = [
    ("sport_type", ["Basketball", "Soccer", "Track", "Other"]),
    ("gender", ["Female", "Male"]),
]
for group_column, group_order in subgroup_specs:
    for group_value in group_order:
        subgroup_frame = residual_diagnostic_df[residual_diagnostic_df[group_column] == group_value].copy()
        X = sm.add_constant(subgroup_frame[["y_t_smooth_past", "u_t"]], has_constant="add")
        result = sm.OLS(subgroup_frame["y_next_smooth_past"], X).fit()
        subgroup_rows.append(
            {
                "group_type": group_column,
                "group": group_value,
                "n_obs": len(subgroup_frame),
                "alpha_hat": result.params["u_t"],
                "rho_hat": result.params["y_t_smooth_past"],
                "k_hat": 1 - result.params["y_t_smooth_past"],
                "r_squared": result.rsquared,
            }
        )
subgroup_transition_table = pd.DataFrame(subgroup_rows)

save_table(smoothing_sensitivity_table, "table_smoothing_sensitivity.csv")
save_table(workload_sensitivity_table, "table_workload_sensitivity.csv")
save_table(model_shape_comparison, "table_model_shape_comparison.csv")
save_table(residual_summary_table, "table_residual_summary.csv")
save_table(residual_by_injury_table, "table_residual_by_injury.csv")
save_table(calibration_table, "table_prediction_calibration.csv")
save_table(subgroup_transition_table, "table_subgroup_transition_parameters.csv")

display(Markdown("### Smoothing sensitivity"))
display(smoothing_sensitivity_table)
display(Markdown("### Workload-definition sensitivity"))
display(workload_sensitivity_table)
display(Markdown("### Model-shape comparison"))
display(model_shape_comparison)
display(Markdown("### Residual diagnostics"))
display(residual_summary_table)
display(residual_by_injury_table)
display(Markdown("### Prediction calibration by decile"))
display(calibration_table)
display(Markdown("### Subgroup-specific ODE parameters"))
subgroup_transition_table


In [ ]:
# Step 5F: visualize three especially important adequacy diagnostics.
fig, axes = plt.subplots(1, 3, figsize=(11.2, 3.2), gridspec_kw={"width_ratios": [1, 1, 1.1]})

sns.barplot(
    data=smoothing_sensitivity_table,
    x="smoothing_window",
    y="alpha_hat",
    color="#4d908e",
    edgecolor="none",
    ax=axes[0],
)
axes[0].axhline(0, color="black", linestyle="--", linewidth=0.9)
axes[0].set_title("Accumulation estimate by smoothing window", pad=6)
axes[0].set_xlabel("Trailing window")
axes[0].set_ylabel("alpha_hat")
for x_pos, (_, row) in enumerate(smoothing_sensitivity_table.iterrows()):
    axes[0].annotate(
        f"k={row['k_hat']:.2f}",
        (x_pos, row["alpha_hat"]),
        ha="center",
        va="bottom" if row["alpha_hat"] >= 0 else "top",
        fontsize=8,
        xytext=(0, 4 if row["alpha_hat"] >= 0 else -10),
        textcoords="offset points",
    )

max_val = max(calibration_table["mean_predicted"].max(), calibration_table["mean_observed"].max())
min_val = min(calibration_table["mean_predicted"].min(), calibration_table["mean_observed"].min())
axes[1].plot([min_val, max_val], [min_val, max_val], color="black", linestyle="--", linewidth=1)
sns.scatterplot(
    data=calibration_table,
    x="mean_predicted",
    y="mean_observed",
    s=48,
    color="#e76f51",
    edgecolor="none",
    ax=axes[1],
)
for _, row in calibration_table.iterrows():
    axes[1].annotate(
        row["prediction_decile"],
        (row["mean_predicted"], row["mean_observed"]),
        fontsize=7,
        xytext=(4, 3),
        textcoords="offset points",
    )
axes[1].set_title("Calibration of ODE transition predictions", pad=6)
axes[1].set_xlabel("Mean predicted next-session y(t)")
axes[1].set_ylabel("Mean observed next-session y(t)")

lag_plot_table = lag_workload_table.copy()
axes[2].errorbar(
    lag_plot_table["lag"],
    lag_plot_table["coefficient"],
    yerr=1.96 * lag_plot_table["std_error"],
    fmt="o-",
    color="#577590",
    ecolor="#577590",
    elinewidth=1,
    capsize=2,
)
axes[2].axhline(0, color="black", linestyle="--", linewidth=0.9)
axes[2].set_title("Lagged workload contributions", pad=6)
axes[2].set_xlabel("Workload lag")
axes[2].set_ylabel("Coefficient")

fig.tight_layout(pad=0.5, w_pad=1.2)
save_figure(fig, "figure_0b_ode_diagnostics.png")
plt.show()


In [ ]:
# Step 5G: interpret the deeper adequacy diagnostics.
nonlinear_candidates = [
    "ODE + quadratic workload",
    "ODE + interaction",
    "ODE + quadratic y and u",
    "ODE + quadratic + interaction",
]
best_nonlinear_row = (
    model_shape_comparison[model_shape_comparison["model"].isin(nonlinear_candidates)]
    .sort_values("rmse")
    .iloc[0]
)
calibration_corr = calibration_table["mean_predicted"].corr(calibration_table["mean_observed"])
injured_abs_resid = residual_by_injury_table.loc[
    residual_by_injury_table["injury_label"] == "Injured",
    "mean_abs_residual",
].iloc[0]
healthy_abs_resid = residual_by_injury_table.loc[
    residual_by_injury_table["injury_label"] == "Healthy",
    "mean_abs_residual",
].iloc[0]

display(
    Markdown(
        f'''
**Evaluation of the adequacy and sensitivity checks**

- The smoothed ODE signs are **not** driven by one arbitrary window choice. For every trailing window from 2 to 7,
  $\hat{{\alpha}}$ remains positive and $\hat{{k}}$ remains positive. The only sign failure occurs at window 1,
  which is exactly the unsmoothed raw series.
- The choice of workload definition is also reasonably stable. Using `training_load` or
  `training_intensity × training_duration` gives the same qualitative result: positive accumulation,
  positive recovery, and similar fit.
- Modest nonlinear extensions help only a little. The best nonlinear benchmark in grouped CV is
  **{best_nonlinear_row["model"]}** with RMSE **{best_nonlinear_row["rmse"]:.3f}**, compared with
  **{ode_shape_row["rmse"]:.3f}** for the simple ODE transition. That is a small gain, not a wholesale failure of the linear form.
- Calibration is strong at the decile level: the correlation between mean predicted and mean observed next-session fatigue is
  **{calibration_corr:.3f}**.
- Residual serial structure is modest after conditioning on persistence and workload:
  the median athlete-level lag-1 residual correlation is **{athlete_residual_lag1.median():.3f}**.
- The model is less accurate on injured sessions than healthy sessions: mean absolute residual is
  **{injured_abs_resid:.3f}** for `Injured` versus **{healthy_abs_resid:.3f}** for `Healthy`.
  That is an important clue that abrupt injury episodes may require a richer observation model or regime extension.
- Subgroup fits remain qualitatively stable across sport and gender, so the ODE signal is not confined to one obvious subgroup.
- Adjusted $R^2$ is almost identical to ordinary $R^2$ in these pooled regressions because the sample is large and the models are small.

**What new issues did these diagnostics uncover?**

The main issue is not that the ODE is wrong. The main issue is that the simple linear one-state model smooths over
high-volatility injured sessions. That supports your planned next steps: latent-state estimation, likelihood-based fitting,
and possibly latent recovery groups or regime structure later on.
        '''
    )
)


## 2C. What Was Still Missing From The Earlier Analysis

Even after the adequacy checks above, two important questions remained:

- Does the model only work as a **one-step local approximation**, or can it track whole athlete trajectories?
- Are the ODE parameters and EM findings **stable enough** to support strong narrative claims?

The next diagnostics address exactly those issues. This is also the clearest answer to "what was still wrong
with the earlier analysis": it leaned too heavily on local fit summaries and did not yet quantify uncertainty
or solution stability.


In [ ]:
# Step 5H: add bootstrap uncertainty and recursive trajectory validation.
#
# The one-step transition is a local model. This cell checks two different things:
# - bootstrap uncertainty: how variable are the key ODE-style coefficients if the
#   data are resampled?
# - recursive validation: what happens when the one-step model is rolled forward
#   repeatedly instead of being refreshed with the observed fatigue state?
#
# That second check is important because a model can work well for one-step updates
# and still drift badly if it is treated like a long-run simulator.
# These checks are important because the project is ultimately about dynamics,
# not just one-step regression fit.

# 1. Cluster bootstrap the athlete-level panels to quantify uncertainty
# in alpha, rho, and k.
bootstrap_rng = np.random.default_rng(RANDOM_STATE)
smoothed_transition_panel = transition_df[
    [athlete_id, time_var, injury_var, "injury_onset", "u_t", "y_t_smooth_past", "y_next_smooth_past"]
].dropna().copy()

bootstrap_athletes = np.array(sorted(smoothed_transition_panel[athlete_id].unique()))
bootstrap_rows = []
for bootstrap_id in range(200):
    sampled_athletes = bootstrap_rng.choice(bootstrap_athletes, size=len(bootstrap_athletes), replace=True)
    bootstrap_frames = []
    for cluster_id, sampled_athlete in enumerate(sampled_athletes):
        athlete_frame = smoothed_transition_panel[smoothed_transition_panel[athlete_id] == sampled_athlete].copy()
        athlete_frame["bootstrap_group"] = cluster_id
        bootstrap_frames.append(athlete_frame)
    bootstrap_frame = pd.concat(bootstrap_frames, ignore_index=True)
    X = sm.add_constant(bootstrap_frame[["y_t_smooth_past", "u_t"]], has_constant="add")
    bootstrap_result = sm.OLS(bootstrap_frame["y_next_smooth_past"], X).fit(
        cov_type="cluster",
        cov_kwds={"groups": bootstrap_frame["bootstrap_group"]},
    )
    bootstrap_rows.append(
        {
            "alpha_hat": bootstrap_result.params["u_t"],
            "rho_hat": bootstrap_result.params["y_t_smooth_past"],
            "k_hat": 1 - bootstrap_result.params["y_t_smooth_past"],
        }
    )
parameter_bootstrap = pd.DataFrame(bootstrap_rows)
parameter_bootstrap_ci = (
    parameter_bootstrap.quantile([0.025, 0.5, 0.975]).T.reset_index()
    .rename(columns={"index": "parameter", 0.025: "ci_lower", 0.5: "median", 0.975: "ci_upper"})
)

# 2. Validate the ODE as an actual recursive dynamic model.
# One-step predictions use the observed current state. Recursive predictions
# instead propagate the model forward through time.
recursive_frames = []
pooled_const = smoothed_transition_result.params["const"]
pooled_rho = smoothed_transition_result.params["y_t_smooth_past"]
pooled_alpha = smoothed_transition_result.params["u_t"]

for athlete, athlete_frame in smoothed_transition_panel.groupby(athlete_id):
    athlete_frame = athlete_frame.sort_values(time_var).copy().reset_index(drop=True)

    one_step_X = sm.add_constant(athlete_frame[["y_t_smooth_past", "u_t"]], has_constant="add")
    athlete_frame["yhat_one_step"] = smoothed_transition_result.predict(one_step_X)

    recursive_state = athlete_frame.loc[0, "y_t_smooth_past"]
    recursive_predictions = []
    persistence_predictions = []
    persistence_state = athlete_frame.loc[0, "y_t_smooth_past"]
    for _, row in athlete_frame.iterrows():
        next_recursive = pooled_const + pooled_rho * recursive_state + pooled_alpha * row["u_t"]
        recursive_predictions.append(next_recursive)
        persistence_predictions.append(persistence_state)
        recursive_state = next_recursive

    athlete_frame["yhat_recursive_ode"] = recursive_predictions
    athlete_frame["yhat_recursive_persistence"] = persistence_predictions
    recursive_frames.append(athlete_frame)

recursive_validation_df = pd.concat(recursive_frames, ignore_index=True)
recursive_validation_table = pd.DataFrame(
    [
        {
            "model": "One-step ODE transition",
            "rmse": mean_squared_error(recursive_validation_df["y_next_smooth_past"], recursive_validation_df["yhat_one_step"]) ** 0.5,
            "mae": mean_absolute_error(recursive_validation_df["y_next_smooth_past"], recursive_validation_df["yhat_one_step"]),
            "r_squared": r2_score(recursive_validation_df["y_next_smooth_past"], recursive_validation_df["yhat_one_step"]),
        },
        {
            "model": "Recursive ODE trajectory",
            "rmse": mean_squared_error(recursive_validation_df["y_next_smooth_past"], recursive_validation_df["yhat_recursive_ode"]) ** 0.5,
            "mae": mean_absolute_error(recursive_validation_df["y_next_smooth_past"], recursive_validation_df["yhat_recursive_ode"]),
            "r_squared": r2_score(recursive_validation_df["y_next_smooth_past"], recursive_validation_df["yhat_recursive_ode"]),
        },
        {
            "model": "Recursive persistence",
            "rmse": mean_squared_error(recursive_validation_df["y_next_smooth_past"], recursive_validation_df["yhat_recursive_persistence"]) ** 0.5,
            "mae": mean_absolute_error(recursive_validation_df["y_next_smooth_past"], recursive_validation_df["yhat_recursive_persistence"]),
            "r_squared": r2_score(recursive_validation_df["y_next_smooth_past"], recursive_validation_df["yhat_recursive_persistence"]),
        },
    ]
)

# 3. Check where the ODE errors concentrate relative to injury onset.
recursive_validation_df["one_step_residual"] = (
    recursive_validation_df["y_next_smooth_past"] - recursive_validation_df["yhat_one_step"]
)
onset_residual_rows = []
for athlete, athlete_frame in recursive_validation_df.groupby(athlete_id):
    athlete_frame = athlete_frame.reset_index(drop=True)
    onset_indices = athlete_frame.index[athlete_frame["injury_onset"]].tolist()
    for onset_idx in onset_indices:
        for rel_session in range(-3, 4):
            j = onset_idx + rel_session
            if 0 <= j < len(athlete_frame):
                onset_residual_rows.append(
                    {
                        "rel_session": rel_session,
                        "residual": athlete_frame.loc[j, "one_step_residual"],
                    }
                )
onset_residual_df = pd.DataFrame(onset_residual_rows)
onset_residual_summary = (
    onset_residual_df.groupby("rel_session")["residual"]
    .agg(mean_residual="mean", median_residual="median", count="size")
    .reset_index()
)

save_table(parameter_bootstrap_ci, "table_parameter_bootstrap_ci.csv")
save_table(recursive_validation_table, "table_recursive_validation.csv")
save_table(onset_residual_summary, "table_onset_residual_pattern.csv")

display(Markdown("### Bootstrap uncertainty for ODE parameters"))
display(parameter_bootstrap_ci)
display(Markdown("### Recursive trajectory validation"))
display(recursive_validation_table)
display(Markdown("### One-step residual pattern around injury onset"))
onset_residual_summary


In [ ]:
# Step 5I: visualize the trajectory-level diagnostics.
fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.2), gridspec_kw={"width_ratios": [0.9, 1.1]})

validation_plot_table = recursive_validation_table.copy()
sns.barplot(
    data=validation_plot_table,
    x="model",
    y="rmse",
    hue="model",
    palette=["#457b9d", "#e76f51", "#adb5bd"],
    dodge=False,
    legend=False,
    edgecolor="none",
    ax=axes[0],
)
axes[0].set_title("Trajectory validation", pad=6)
axes[0].set_xlabel("")
axes[0].set_ylabel("RMSE")
axes[0].tick_params(axis="x", rotation=16)
for patch, value in zip(axes[0].patches, validation_plot_table["rmse"]):
    axes[0].annotate(
        f"{value:.2f}",
        (patch.get_x() + patch.get_width() / 2, patch.get_height()),
        ha="center",
        va="bottom",
        fontsize=8,
        xytext=(0, 2),
        textcoords="offset points",
    )

sns.lineplot(
    data=onset_residual_summary,
    x="rel_session",
    y="mean_residual",
    marker="o",
    color="#d62828",
    linewidth=2,
    ax=axes[1],
)
axes[1].axhline(0, color="black", linestyle="--", linewidth=0.9)
axes[1].axvline(0, color="black", linestyle="--", linewidth=0.9)
axes[1].set_title("ODE residuals around injury onset", pad=6)
axes[1].set_xlabel("Sessions relative to onset")
axes[1].set_ylabel("Observed minus predicted next y(t)")

fig.tight_layout(pad=0.5, w_pad=1.0)
save_figure(fig, "figure_0c_trajectory_validation.png")
plt.show()


In [ ]:
# Step 5J: interpret the trajectory-level diagnostics.
alpha_ci = parameter_bootstrap_ci.loc[parameter_bootstrap_ci["parameter"] == "alpha_hat"].iloc[0]
k_ci = parameter_bootstrap_ci.loc[parameter_bootstrap_ci["parameter"] == "k_hat"].iloc[0]
recursive_ode_row = recursive_validation_table.loc[
    recursive_validation_table["model"] == "Recursive ODE trajectory"
].iloc[0]
one_step_row = recursive_validation_table.loc[
    recursive_validation_table["model"] == "One-step ODE transition"
].iloc[0]
recursive_persistence_row = recursive_validation_table.loc[
    recursive_validation_table["model"] == "Recursive persistence"
].iloc[0]
onset_pre_row = onset_residual_summary.loc[onset_residual_summary["rel_session"] == -1].iloc[0]
onset_post_row = onset_residual_summary.loc[onset_residual_summary["rel_session"] == 2].iloc[0]

display(
    Markdown(
        f'''
**Evaluation of the uncertainty and trajectory-level diagnostics**

- The pooled smoothed-transition estimates are reasonably tight under cluster bootstrap:
  $\hat{{\alpha}}$ has a 95% interval from **{alpha_ci['ci_lower']:.4f}** to **{alpha_ci['ci_upper']:.4f}**,
  and $\hat{{k}}$ has a 95% interval from **{k_ci['ci_lower']:.3f}** to **{k_ci['ci_upper']:.3f}**.
- The model is much stronger as a **one-step local law** than as a full recursive trajectory model.
  One-step ODE fit has RMSE **{one_step_row['rmse']:.3f}**, while recursive ODE fit drops to RMSE
  **{recursive_ode_row['rmse']:.3f}**.
- Even so, the recursive ODE still strongly outperforms a naive recursive persistence model
  (RMSE **{recursive_persistence_row['rmse']:.3f}**), so it retains meaningful dynamic information.
- Residuals around injury onset are highly structured. One session before onset, the mean one-step residual is
  **{onset_pre_row['mean_residual']:.3f}**, meaning the model underpredicts the pre-onset fatigue rise.
  Two sessions after onset, the mean residual is **{onset_post_row['mean_residual']:.3f}**, meaning the model
  overpredicts fatigue after the post-onset drop.

**What was wrong with the earlier analysis?**

The earlier notebook showed that the ODE worked locally, but it did not yet show these two harder facts:
1. uncertainty on the pooled parameters is fairly small, and
2. the main failure mode is a structured onset/recovery shock that the simple one-state model smooths over.
That is a much more precise limitation than simply saying "the model is imperfect."
        '''
    )
)


In [ ]:
# Step 6: create a core summary table by injury category.
# This is one of the main tables that can go directly into the report.
group_summary = (
    df.groupby(injury_var)
    .agg(
        mean_workload=("u_t", "mean"),
        mean_fatigue_proxy=("y_t", "mean"),
        mean_recovery_score=("recovery_score", "mean"),
        sessions=("u_t", "size"),
    )
    .reset_index()
)
group_summary["injury_label"] = group_summary[injury_var].map(INJURY_LABELS)
group_summary = group_summary[
    ["injury_label", "mean_workload", "mean_fatigue_proxy", "mean_recovery_score", "sessions"]
]

# Also compute standardized differences between injured and non-injured sessions.
def standardized_mean_diff(group_a: pd.Series, group_b: pd.Series) -> float:
    a = pd.to_numeric(group_a, errors="coerce").dropna()
    b = pd.to_numeric(group_b, errors="coerce").dropna()
    if len(a) < 2 or len(b) < 2:
        return np.nan
    pooled_var = (((len(a) - 1) * a.var(ddof=1)) + ((len(b) - 1) * b.var(ddof=1))) / (len(a) + len(b) - 2)
    if pooled_var <= 0:
        return np.nan
    return (a.mean() - b.mean()) / np.sqrt(pooled_var)

injured_df = df[df[injury_var] == 2]
non_injured_df = df[df[injury_var] != 2]
effect_candidates = [
    "u_t",
    "y_t",
    "recovery_score",
    "sleep_quality",
    "training_intensity",
    "training_duration",
    "stress_level",
    "hydration_level",
]
effect_table = pd.DataFrame(
    [
        {
            "variable": column,
            "injured_mean": injured_df[column].mean(),
            "non_injured_mean": non_injured_df[column].mean(),
            "std_mean_diff": standardized_mean_diff(injured_df[column], non_injured_df[column]),
        }
        for column in effect_candidates
    ]
).sort_values("std_mean_diff", key=lambda s: s.abs(), ascending=False)

save_table(group_summary, "table_injury_category_summary.csv")
save_table(effect_table, "table_effect_sizes_injured_vs_noninjured.csv")

display(Markdown("### Injury-category summary table"))
display(group_summary)
display(Markdown("### Effect sizes: injured vs non-injured"))
effect_table


In [ ]:
# Interpretation of the first result table.
workload_proxy_corr = df[["u_t", "u_intensity_x_duration"]].corr().iloc[0, 1]
workload_fatigue_corr = df[["u_t", "y_t"]].corr().iloc[0, 1]

display(
    Markdown(
        f'''
**Evaluation of the core summary tables**

- `training_load` is a defensible workload input because it correlates **{workload_proxy_corr:.3f}**
  with the simple proxy `training_intensity × training_duration`.
- `u(t)` and `y(t)` are strongly aligned overall with correlation **{workload_fatigue_corr:.3f}**.
- Injured sessions have the highest mean workload (**{group_summary.loc[group_summary['injury_label'] == 'Injured', 'mean_workload'].iloc[0]:.1f}**)
  and the highest mean fatigue proxy (**{group_summary.loc[group_summary['injury_label'] == 'Injured', 'mean_fatigue_proxy'].iloc[0]:.1f}**).
- The strongest injured-vs-non-injured contrasts are lower `recovery_score`, lower `sleep_quality`,
  and higher `fatigue_index`, which is consistent with a fatigue/recovery interpretation.

**Are we on the right track?**

Yes for the basic ODE setup. These tables support the choices of `training_load` for $u(t)$ and
`fatigue_index` for $y(t)$, and they show meaningful separation by injury status.
        '''
    )
)


In [ ]:
# Step 7: build Figure 1.
# Left panel = workload by injury category.
# Right panel = athlete-level mean workload vs athlete-level injury rate.
athlete_summary = (
    df.groupby(athlete_id)
    .agg(
        mean_workload=("u_t", "mean"),
        injury_rate=(injury_var, lambda s: (s > 0).mean()),
    )
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.2), gridspec_kw={"width_ratios": [1, 1.1]})

sns.barplot(
    data=group_summary,
    x="injury_label",
    y="mean_workload",
    hue="injury_label",
    palette=["#9ecae1", "#6baed6", "#08519c"],
    dodge=False,
    legend=False,
    edgecolor="none",
    ax=axes[0],
)
axes[0].set_title("Workload by injury category", pad=6)
axes[0].set_xlabel("")
axes[0].set_ylabel("Mean workload")
axes[0].tick_params(axis="x", rotation=15)
for patch, value in zip(axes[0].patches, group_summary["mean_workload"]):
    axes[0].annotate(
        f"{value:.1f}",
        (patch.get_x() + patch.get_width() / 2, patch.get_height()),
        ha="center",
        va="bottom",
        fontsize=8,
        xytext=(0, 2),
        textcoords="offset points",
    )

sns.regplot(
    data=athlete_summary,
    x="mean_workload",
    y="injury_rate",
    ci=None,
    scatter_kws={"s": 18, "alpha": 0.65, "color": "#1d3557", "linewidths": 0},
    line_kws={"color": "#d62828", "linewidth": 1.5},
    ax=axes[1],
)
axes[1].set_title("Athlete mean workload vs injury rate", pad=6)
axes[1].set_xlabel("Mean workload")
axes[1].set_ylabel("Share with injury > 0")

fig.tight_layout(pad=0.5, w_pad=1.0)
save_figure(fig, "figure_1_workload_injury_summary.png")
plt.show()


In [ ]:
# Evaluate Figure 1 numerically.
athlete_level_corr = athlete_summary[["mean_workload", "injury_rate"]].corr().iloc[0, 1]
display(
    Markdown(
        f'''
**Evaluation of Figure 1**

- The left panel confirms that the `Injured` group has the highest mean workload.
- The `Low Risk` group is slightly below `Healthy`, so the pattern is not strictly monotone across all three classes.
- The athlete-level correlation between mean workload and injury rate is **{athlete_level_corr:.3f}**.

**Are we on the right track?**

Yes, but with an important nuance: the strongest signal is the contrast between `Injured` and the other groups,
not a perfectly graded ladder from Healthy to Low Risk to Injured.
        '''
    )
)


In [ ]:
# Step 8: build Figure 2 with compact marginal distributions.
fig, axes = plt.subplots(1, 3, figsize=(8.2, 2.8), gridspec_kw={"width_ratios": [1, 1, 0.92]})

sns.histplot(df["u_t"], bins=28, kde=True, color="#2a9d8f", line_kws={"linewidth": 1.2}, ax=axes[0])
axes[0].set_title("Workload distribution", pad=6)
axes[0].set_xlabel("u(t)")
axes[0].set_ylabel("Count")

sns.histplot(df["y_t"], bins=28, kde=True, color="#e76f51", line_kws={"linewidth": 1.2}, ax=axes[1])
axes[1].set_title("Fatigue proxy distribution", pad=6)
axes[1].set_xlabel("y(t)")
axes[1].set_ylabel("")

injury_counts = (
    df[injury_var]
    .map(INJURY_LABELS)
    .value_counts()
    .reindex(list(INJURY_LABELS.values()), fill_value=0)
    .reset_index()
)
injury_counts.columns = ["injury_label", "count"]
sns.barplot(
    data=injury_counts,
    x="injury_label",
    y="count",
    hue="injury_label",
    palette=["#adb5bd", "#74c69d", "#e63946"],
    dodge=False,
    legend=False,
    edgecolor="none",
    ax=axes[2],
)
axes[2].set_title("Injury category counts", pad=6)
axes[2].set_xlabel("")
axes[2].set_ylabel("")
axes[2].tick_params(axis="x", rotation=15)

fig.tight_layout(pad=0.4, w_pad=0.8)
save_figure(fig, "figure_2_distributions.png")
plt.show()


In [ ]:
# Evaluate Figure 2.
injury_mix = (100 * df[injury_var].value_counts(normalize=True).sort_index()).round(1)
display(
    Markdown(
        f'''
**Evaluation of Figure 2**

- The injury outcome is imbalanced: Healthy = **{injury_mix.loc[0]:.1f}%**, Low Risk = **{injury_mix.loc[1]:.1f}%**, Injured = **{injury_mix.loc[2]:.1f}%**.
- Both workload and fatigue proxy are continuous and broadly distributed, which supports modeling them as signals rather than binary flags.
- The distributions are not degenerate, so there is enough variation for a latent-state model to be meaningful.

**Are we on the right track?**

Yes. The dataset has enough continuous structure in workload and fatigue to motivate dynamic modeling, while the injury outcome remains a stratifying label rather than the only variable of interest.
        '''
    )
)


In [ ]:
# Step 9: compute dynamic summary tables that are especially relevant for the ODE idea.
#
# These summaries are less formal than the regression tables above, but they are
# still part of the argument. They show whether the workload-fatigue relationship
# and the injury gradients line up with the basic story the model is trying to tell.
# If smoothed fatigue stratifies later injury more clearly than current workload,
# that supports the decision to model internal state rather than exposure alone.
def athlete_correlation_table(df: pd.DataFrame, athlete_id: str) -> pd.DataFrame:
    rows = []
    for athlete, athlete_df in df.groupby(athlete_id):
        rows.append(
            {
                athlete_id: athlete,
                "corr_u_y_same": athlete_df["u_t"].corr(athlete_df["y_t"]),
                "corr_u_y_next": athlete_df["u_t"].corr(athlete_df["y_t"].shift(-1)),
                "corr_u_y_smooth": athlete_df["u_t"].corr(athlete_df["y_t_smooth"]),
            }
        )
    return pd.DataFrame(rows)


def quartile_risk_table(df: pd.DataFrame, metric_map: dict[str, str]) -> pd.DataFrame:
    tables = []
    for metric, label in metric_map.items():
        tmp = df[[metric, "next_injured"]].dropna().copy()
        tmp["quartile"] = pd.qcut(tmp[metric], 4, labels=["Q1", "Q2", "Q3", "Q4"])
        summary = (
            tmp.groupby("quartile", observed=False)["next_injured"]
            .agg(next_injured_sessions="sum", sessions="count", risk="mean")
            .reset_index()
        )
        summary.insert(0, "metric", label)
        tables.append(summary)
    return pd.concat(tables, ignore_index=True)


def onset_window_summary(df: pd.DataFrame, athlete_id: str, time_var: str, window: int = 3):
    rows = []
    onset_count = 0
    for athlete, athlete_df in df.groupby(athlete_id):
        athlete_df = athlete_df.reset_index(drop=True)
        onset_indices = athlete_df.index[athlete_df["injury_onset"]].tolist()
        onset_count += len(onset_indices)
        for idx in onset_indices:
            for rel in range(-window, window + 1):
                j = idx + rel
                if 0 <= j < len(athlete_df):
                    rows.append(
                        {
                            athlete_id: athlete,
                            "rel_session": rel,
                            "u_t": athlete_df.loc[j, "u_t"],
                            "y_t": athlete_df.loc[j, "y_t"],
                        }
                    )
    onset_df = pd.DataFrame(rows)
    summary = (
        onset_df.groupby("rel_session")[["u_t", "y_t"]]
        .agg(["mean", "median"])
        .reset_index()
    )
    summary.columns = ["rel_session", "u_t_mean", "u_t_median", "y_t_mean", "y_t_median"]
    return summary, onset_count


athlete_corr = athlete_correlation_table(df, athlete_id)
athlete_corr_summary = athlete_corr.drop(columns=[athlete_id]).describe().T.reset_index().rename(columns={"index": "metric"})

risk_table = quartile_risk_table(
    df,
    {
        "u_t": "Current workload u(t)",
        "u_ma_3": "3-session rolling workload",
        "y_t_smooth": "Smoothed fatigue proxy y(t)",
    },
)
risk_table["risk_pct"] = 100 * risk_table["risk"]

risk_ratio_table = (
    risk_table.pivot(index="quartile", columns="metric", values="risk")
    .rename_axis(index=None, columns=None)
)
risk_ratio_table = pd.DataFrame(
    {
        "metric": risk_ratio_table.columns,
        "q4_vs_q1_risk_ratio": [risk_ratio_table.loc["Q4", column] / risk_ratio_table.loc["Q1", column] for column in risk_ratio_table.columns],
    }
)

onset_summary, onset_count = onset_window_summary(df, athlete_id, time_var, window=3)

save_table(athlete_corr, "table_athlete_correlations.csv")
save_table(athlete_corr_summary, "table_athlete_correlation_summary.csv")
save_table(risk_table, "table_next_session_risk_by_quartile.csv")
save_table(risk_ratio_table, "table_next_session_risk_ratios.csv")
save_table(onset_summary, "table_injury_onset_window_summary.csv")

display(Markdown("### Athlete-specific correlations"))
display(athlete_corr_summary)
display(Markdown("### Next-session injury risk by quartile"))
display(risk_table[["metric", "quartile", "next_injured_sessions", "sessions", "risk_pct"]])
display(Markdown("### Highest vs lowest quartile risk ratios"))
display(risk_ratio_table)
display(Markdown(f"### Injury-onset window summary (n = {onset_count} onset sessions)"))
onset_summary


In [ ]:
# Evaluate the dynamic tables before moving to the dynamic figures.
same_corr_median = athlete_corr["corr_u_y_same"].median()
next_corr_median = athlete_corr["corr_u_y_next"].median()
smooth_corr_median = athlete_corr["corr_u_y_smooth"].median()

display(
    Markdown(
        f'''
**Evaluation of the dynamic summary tables**

- The median within-athlete same-session workload-fatigue correlation is **{same_corr_median:.3f}**.
- The median within-athlete lagged correlation between current workload and next-session fatigue is **{next_corr_median:.3f}**.
- The median within-athlete correlation between workload and smoothed fatigue is **{smooth_corr_median:.3f}**.
- The highest quartile of smoothed fatigue burden has a **{risk_ratio_table.loc[risk_ratio_table['metric'] == 'Smoothed fatigue proxy y(t)', 'q4_vs_q1_risk_ratio'].iloc[0]:.2f}x**
  next-session injury risk relative to the lowest quartile.
- The highest quartile of 3-session rolling workload has a **{risk_ratio_table.loc[risk_ratio_table['metric'] == '3-session rolling workload', 'q4_vs_q1_risk_ratio'].iloc[0]:.2f}x**
  next-session injury risk relative to the lowest quartile.

**Are we on the right track?**

Yes, and this is one of the strongest parts of the notebook. These tables move the argument beyond cross-sectional separation and show clear dynamic burden patterns.
        '''
    )
)


In [ ]:
# Step 10: derivative-based diagnostic figure.
# This checks whether a simple gradient-matching approximation looks stable.
gradient_df = df[["u_t", "y_t_smooth", "dy_dt", injury_var]].dropna().copy()
gradient_plot_df = gradient_df.sample(min(3000, len(gradient_df)), random_state=RANDOM_STATE)
gradient_plot_df["injury_label"] = gradient_plot_df[injury_var].map(INJURY_LABELS)

palette = {"Healthy": "#577590", "Low Risk": "#43aa8b", "Injured": "#f94144"}
fig, axes = plt.subplots(1, 2, figsize=(8.3, 3.0))

sns.scatterplot(
    data=gradient_plot_df,
    x="u_t",
    y="dy_dt",
    hue="injury_label",
    palette=palette,
    s=13,
    alpha=0.38,
    linewidth=0,
    ax=axes[0],
)
sns.regplot(
    data=gradient_plot_df,
    x="u_t",
    y="dy_dt",
    scatter=False,
    truncate=False,
    ci=None,
    line_kws={"color": "black", "linewidth": 1.4},
    ax=axes[0],
)
axes[0].set_title("dy/dt vs workload", pad=6)
axes[0].set_xlabel("u(t)")
axes[0].set_ylabel("dy/dt")

sns.scatterplot(
    data=gradient_plot_df,
    x="y_t_smooth",
    y="dy_dt",
    hue="injury_label",
    palette=palette,
    s=13,
    alpha=0.38,
    linewidth=0,
    legend=False,
    ax=axes[1],
)
sns.regplot(
    data=gradient_plot_df,
    x="y_t_smooth",
    y="dy_dt",
    scatter=False,
    truncate=False,
    ci=None,
    line_kws={"color": "black", "linewidth": 1.4},
    ax=axes[1],
)
axes[1].set_title("dy/dt vs smoothed fatigue", pad=6)
axes[1].set_xlabel("Smoothed y(t)")
axes[1].set_ylabel("")

axes[0].legend(title="Category", loc="upper left", frameon=False, borderaxespad=0.2)
fig.tight_layout(pad=0.5, w_pad=0.9)
save_figure(fig, "figure_3_derivative_analysis.png")
plt.show()


In [ ]:
# Evaluate the derivative-based figure.
corr_u_dydt = gradient_df[["u_t", "dy_dt"]].corr().iloc[0, 1]
corr_y_dydt = gradient_df[["y_t_smooth", "dy_dt"]].corr().iloc[0, 1]

display(
    Markdown(
        f'''
**Evaluation of Figure 3**

- Correlation between `dy/dt` and `u(t)` is **{corr_u_dydt:.3f}**.
- Correlation between `dy/dt` and smoothed `y(t)` is **{corr_y_dydt:.3f}**.
- These are **bivariate** diagnostics only, so they do not enforce the ODE structure directly.
- The formal multivariable regression in the model-justification section above is the correct test; there,
  the workload term is positive and the recovery term is negative on the past-smoothed series.

**Are we on the right track?**

Yes, but this figure is also a warning sign. Finite-difference plots are informative as diagnostics,
yet too noisy to be the only estimation strategy. That supports the plan to keep likelihood-based ODE fitting central.
        '''
    )
)


In [ ]:
# Step 11: plot the quartile-based next-session risk gradients.
plot_risk = risk_table[risk_table["metric"].isin(["3-session rolling workload", "Smoothed fatigue proxy y(t)"])].copy()
plot_risk["quartile"] = pd.Categorical(plot_risk["quartile"], categories=["Q1", "Q2", "Q3", "Q4"], ordered=True)

fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.0), sharey=True)
subset_specs = [
    ("3-session rolling workload", "Recent workload burden", "#457b9d"),
    ("Smoothed fatigue proxy y(t)", "Smoothed fatigue burden", "#e76f51"),
]

for ax, (metric_name, title, color) in zip(axes, subset_specs):
    subset = plot_risk[plot_risk["metric"] == metric_name]
    sns.barplot(data=subset, x="quartile", y="risk_pct", color=color, edgecolor="none", ax=ax)
    ax.set_title(title, pad=6)
    ax.set_xlabel("Quartile")
    ax.set_ylabel("Next-session injured (%)" if ax is axes[0] else "")
    for patch, value in zip(ax.patches, subset["risk_pct"]):
        ax.annotate(
            f"{value:.1f}",
            (patch.get_x() + patch.get_width() / 2, patch.get_height()),
            ha="center",
            va="bottom",
            fontsize=8,
            xytext=(0, 2),
            textcoords="offset points",
        )

fig.tight_layout(pad=0.5, w_pad=0.8)
save_figure(fig, "figure_4_next_session_risk.png")
plt.show()


In [ ]:
# Evaluate the quartile-risk figure.
workload_q1 = plot_risk.query("metric == '3-session rolling workload' and quartile == 'Q1'")["risk_pct"].iloc[0]
workload_q4 = plot_risk.query("metric == '3-session rolling workload' and quartile == 'Q4'")["risk_pct"].iloc[0]
fatigue_q1 = plot_risk.query("metric == 'Smoothed fatigue proxy y(t)' and quartile == 'Q1'")["risk_pct"].iloc[0]
fatigue_q4 = plot_risk.query("metric == 'Smoothed fatigue proxy y(t)' and quartile == 'Q4'")["risk_pct"].iloc[0]

display(
    Markdown(
        f'''
**Evaluation of Figure 4**

- Next-session injured risk rises from **{workload_q1:.1f}%** in the lowest quartile of 3-session workload
  to **{workload_q4:.1f}%** in the highest quartile.
- Next-session injured risk rises from **{fatigue_q1:.1f}%** in the lowest quartile of smoothed fatigue burden
  to **{fatigue_q4:.1f}%** in the highest quartile.
- The fatigue-burden gradient is steeper than the workload-burden gradient.

**Are we on the right track?**

Very much yes. This figure is one of the strongest reasons to continue with the latent-state ODE framing,
because it suggests that accumulated burden matters more than single-session load alone.
        '''
    )
)


In [ ]:
# Step 12: plot mean workload and fatigue proxy around injury onset.
fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.1), sharex=True)

sns.lineplot(data=onset_summary, x="rel_session", y="u_t_mean", marker="o", color="#1d3557", linewidth=2, ax=axes[0])
axes[0].axvline(0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Workload around injury onset", pad=6)
axes[0].set_xlabel("Sessions relative to onset")
axes[0].set_ylabel("Mean u(t)")

sns.lineplot(data=onset_summary, x="rel_session", y="y_t_mean", marker="o", color="#d62828", linewidth=2, ax=axes[1])
axes[1].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Fatigue proxy around injury onset", pad=6)
axes[1].set_xlabel("Sessions relative to onset")
axes[1].set_ylabel("Mean y(t)")

fig.tight_layout(pad=0.5, w_pad=0.9)
save_figure(fig, "figure_5_injury_onset_window.png")
plt.show()


In [ ]:
# Evaluate the onset-window figure with a few anchor numbers.
onset_minus1 = onset_summary.loc[onset_summary["rel_session"] == -1].squeeze()
onset_zero = onset_summary.loc[onset_summary["rel_session"] == 0].squeeze()
onset_plus1 = onset_summary.loc[onset_summary["rel_session"] == 1].squeeze()

display(
    Markdown(
        f'''
**Evaluation of Figure 5**

- Mean workload rises from **{onset_minus1['u_t_mean']:.1f}** one session before onset to **{onset_zero['u_t_mean']:.1f}** at onset.
- Mean fatigue proxy rises from **{onset_minus1['y_t_mean']:.1f}** one session before onset to **{onset_zero['y_t_mean']:.1f}** at onset.
- Both measures drop immediately after onset, with fatigue proxy falling to **{onset_plus1['y_t_mean']:.1f}** one session later.

**Are we on the right track?**

Yes. This pattern is exactly the kind of buildup-and-release structure that makes a recovery-dynamics interpretation plausible.
        '''
    )
)


In [ ]:
# Step 13: show a couple of athlete trajectories so the within-athlete
# temporal co-movement is visible and not just hidden in averages.
athlete_rank = (
    df.groupby(athlete_id)
    .agg(
        observations=("u_t", "size"),
        elevated_risk_sessions=(injury_var, lambda s: (s > 0).sum()),
    )
    .sort_values(["elevated_risk_sessions", "observations"], ascending=[False, False])
)
sample_athletes = athlete_rank.head(2).index.tolist()

fig, axes = plt.subplots(len(sample_athletes), 1, figsize=(8.0, 4.8), sharex=False)
if len(sample_athletes) == 1:
    axes = [axes]

workload_color = "#1d3557"
fatigue_color = "#d62828"
low_risk_color = "#f4a261"
injured_color = "#e63946"

for axis, athlete in zip(axes, sample_athletes):
    athlete_df = df[df[athlete_id] == athlete]
    x_values = athlete_df[time_var]

    axis.plot(x_values, athlete_df["u_t"], color=workload_color, linewidth=1.5)
    axis.set_ylabel("u(t)", color=workload_color)
    axis.tick_params(axis="y", colors=workload_color, length=3)
    axis.set_title(f"Athlete {athlete}", loc="left", pad=4)

    twin_axis = axis.twinx()
    twin_axis.plot(x_values, athlete_df["y_t"], color=fatigue_color, linewidth=1.5, alpha=0.9)
    twin_axis.set_ylabel("y(t)", color=fatigue_color)
    twin_axis.tick_params(axis="y", colors=fatigue_color, length=3)

    low_risk = athlete_df[athlete_df[injury_var] == 1]
    injured = athlete_df[athlete_df[injury_var] == 2]
    if not low_risk.empty:
        axis.scatter(low_risk[time_var], low_risk["u_t"], s=22, color=low_risk_color, marker="o", zorder=4)
    if not injured.empty:
        axis.scatter(injured[time_var], injured["u_t"], s=28, color=injured_color, marker="x", linewidths=1.0, zorder=5)

    axis.set_xlabel("Session index")

legend_handles = [
    Line2D([0], [0], color=workload_color, lw=1.8, label="u(t)"),
    Line2D([0], [0], color=fatigue_color, lw=1.8, label="y(t)"),
    Line2D([0], [0], marker="o", linestyle="", markersize=5, color=low_risk_color, label="Low Risk"),
    Line2D([0], [0], marker="x", linestyle="", markersize=6, color=injured_color, label="Injured"),
]
fig.legend(
    handles=legend_handles,
    loc="upper center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, 1.01),
    handletextpad=0.4,
    columnspacing=1.0,
)

fig.tight_layout(pad=0.55, h_pad=1.0)
save_figure(fig, "figure_6_sample_trajectories.png")
plt.show()


In [ ]:
# Evaluate the sample trajectory figure.
prop_same_gt_075 = (athlete_corr["corr_u_y_same"] > 0.75).mean()
display(
    Markdown(
        f'''
**Evaluation of Figure 6**

- The example athletes show visible within-athlete co-movement between workload and fatigue proxy.
- Risk-coded sessions tend to occur in segments where these two signals are elevated together.
- Across all athletes, **{100 * prop_same_gt_075:.1f}%** have a same-session workload-fatigue correlation above 0.75.

**Are we on the right track?**

Yes. The trajectory plots visually support what the athlete-level correlation table already suggested:
the overall signal is not being driven by a few unusual athletes.
        '''
    )
)


In [ ]:
# Step 14: focused heatmap for the variables most relevant to the model story.
correlation_columns = [
    "u_t",
    "y_t",
    injury_var,
    "training_intensity",
    "training_duration",
    "training_load",
    "recovery_score",
    "heart_rate",
    "sleep_quality",
    "stress_level",
    "hydration_level",
    "muscle_activity",
]
corr_matrix = df[correlation_columns].corr()
corr_matrix = corr_matrix.rename(
    index={
        "u_t": "u(t)",
        "y_t": "y(t)",
        injury_var: "injury_code",
        "training_intensity": "train_intensity",
        "training_duration": "train_duration",
        "training_load": "train_load",
        "recovery_score": "recovery",
        "heart_rate": "heart_rate",
        "sleep_quality": "sleep_quality",
        "stress_level": "stress",
        "hydration_level": "hydration",
        "muscle_activity": "muscle_activity",
    },
    columns={
        "u_t": "u(t)",
        "y_t": "y(t)",
        injury_var: "injury_code",
        "training_intensity": "train_intensity",
        "training_duration": "train_duration",
        "training_load": "train_load",
        "recovery_score": "recovery",
        "heart_rate": "heart_rate",
        "sleep_quality": "sleep_quality",
        "stress_level": "stress",
        "hydration_level": "hydration",
        "muscle_activity": "muscle_activity",
    },
)

fig, ax = plt.subplots(figsize=(6.5, 5.4))
sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
    fmt=".1f",
    annot_kws={"size": 7},
    linewidths=0.3,
    cbar_kws={"shrink": 0.82, "pad": 0.02},
    ax=ax,
)
ax.set_title("Correlation heatmap\n(injury code is descriptive)", pad=8)
ax.tick_params(axis="x", rotation=40)
ax.tick_params(axis="y", rotation=0)

fig.tight_layout(pad=0.4)
save_figure(fig, "figure_7_correlation_heatmap.png")
plt.show()


In [ ]:
# Evaluate the heatmap using a few especially relevant pairwise relationships.
display(
    Markdown(
        f'''
**Evaluation of Figure 7**

- Correlation between `u(t)` and `y(t)` is **{corr_matrix.loc['u(t)', 'y(t)']:.1f}**.
- Correlation between `y(t)` and `recovery` is **{corr_matrix.loc['y(t)', 'recovery']:.1f}**.
- Correlation between `y(t)` and `injury_code` is **{corr_matrix.loc['y(t)', 'injury_code']:.1f}**.

**Are we on the right track?**

Yes, with one caution: this heatmap is descriptive, and `injury_code` is not a true continuous variable.
It is useful for seeing broad structure, but it should not replace the more targeted dynamic summaries above.
        '''
    )
)


## 4. EM-Centered Latent Recovery Profile Analysis

The earlier ODE sections answered a preliminary question: does the data support a simple accumulation-and-recovery
mechanism strongly enough that athlete-level dynamic summaries are meaningful? This section uses those summaries
as **stage-1 features** for the main latent-profile analysis.

The primary EM feature set is intentionally small and justified:

- $\hat{\alpha}$: athlete-specific workload sensitivity.
- estimated recovery half-life: an interpretable version of the recovery-rate parameter.
- transition $R^2$: how consistently the simple ODE transition explains that athlete's local fatigue dynamics.
- injured-session rate: empirical burden / outcome anchor.
- mean recovery score: sustained recovery context.

Other variables in the dataset are still used, but mainly as **withheld validation variables** rather than direct
clustering inputs. That includes sleep, workload, stress, hydration, physiology, onset-based recovery summaries,
age, BMI, sport, and gender. This keeps the mixture focused on persistent recovery phenotype rather than letting
noisy session-level context dominate the clustering.


In [ ]:
# Step 15: build an athlete-level feature matrix for the EM analysis.
#
# This is where the analysis switches from session-level dynamics to athlete-level
# latent profiling.
#
# The mixture model should not cluster raw session rows. The scientific question is
# whether athletes differ in their *recovery pattern*, so the clustering unit must
# be the athlete. That means each input to EM has to be turned into a stable summary
# over that athlete's panel.
#
# The primary feature set is intentionally narrow:
# - alpha_hat: workload sensitivity,
# - half_life_sessions: recovery time scale,
# - r_squared: coherence of the simple transition model,
# - injury_rate_injured: burden anchor,
# - mean_recovery_score: sustained recovery context.
#
# Additional variables are still collected here, but most of them are held back for
# validation. That separation is deliberate: it lets the analysis ask whether the
# latent groups discovered from recovery features also differ on outside variables,
# instead of forcing those outside variables into the cluster definition from the start.
# The clustering step should operate on stable athlete summaries, not raw session rows.
athlete_validation = (
    df.groupby(athlete_id)
    .agg(
        mean_stress_level=("stress_level", "mean"),
        mean_hydration=("hydration_level", "mean"),
        mean_heart_rate=("heart_rate", "mean"),
        mean_muscle_activity=("muscle_activity", "mean"),
        mean_jump_height=("jump_height", "mean"),
        mean_gait_speed=("gait_speed", "mean"),
        mean_range_of_motion=("range_of_motion", "mean"),
        age=("age", "first"),
        bmi=("bmi", "first"),
    )
    .reset_index()
)

# Build onset-based recovery summaries that will be used later for external validation.
onset_rows = []
for athlete, athlete_df in df.groupby(athlete_id):
    athlete_df = athlete_df.reset_index(drop=True)
    onset_indices = athlete_df.index[athlete_df["injury_onset"]].tolist()
    per_onset_rows = []
    for idx in onset_indices:
        if idx + 5 < len(athlete_df):
            per_onset_rows.append(
                {
                    "onset_y": athlete_df.loc[idx, "y_t"],
                    "drop_y_1": athlete_df.loc[idx, "y_t"] - athlete_df.loc[idx + 1, "y_t"],
                    "drop_y_3": athlete_df.loc[idx, "y_t"] - athlete_df.loc[idx + 3, "y_t"],
                    "drop_y_5": athlete_df.loc[idx, "y_t"] - athlete_df.loc[idx + 5, "y_t"],
                    "rec_post3": athlete_df.loc[idx + 1 : idx + 3, "recovery_score"].mean(),
                    "rec_post5": athlete_df.loc[idx + 1 : idx + 5, "recovery_score"].mean(),
                    "u_drop_3": athlete_df.loc[idx, "u_t"] - athlete_df.loc[idx + 3, "u_t"],
                }
            )

    athlete_features = {athlete_id: athlete, "onset_count": len(per_onset_rows)}
    if per_onset_rows:
        onset_frame = pd.DataFrame(per_onset_rows)
        athlete_features.update(onset_frame.mean().to_dict())
        athlete_features["drop_y_1_sd"] = onset_frame["drop_y_1"].std(ddof=0)
        athlete_features["drop_y_3_sd"] = onset_frame["drop_y_3"].std(ddof=0)
        athlete_features["drop_y_5_sd"] = onset_frame["drop_y_5"].std(ddof=0)
    else:
        athlete_features.update(
            {
                "onset_y": np.nan,
                "drop_y_1": np.nan,
                "drop_y_3": np.nan,
                "drop_y_5": np.nan,
                "rec_post3": np.nan,
                "rec_post5": np.nan,
                "u_drop_3": np.nan,
                "drop_y_1_sd": np.nan,
                "drop_y_3_sd": np.nan,
                "drop_y_5_sd": np.nan,
            }
        )
    onset_rows.append(athlete_features)

em_onset_features = pd.DataFrame(onset_rows)

em_primary_frame = (
    athlete_transition_detail.merge(em_onset_features, on=athlete_id, how="left")
    .merge(athlete_validation, on=athlete_id, how="left")
)

primary_feature_columns = [
    "alpha_hat",
    "half_life_sessions",
    "r_squared",
    "injury_rate_injured",
    "mean_recovery_score",
]
validation_feature_columns = [
    "mean_sleep_quality",
    "mean_workload",
    "mean_stress_level",
    "mean_hydration",
    "mean_heart_rate",
    "mean_muscle_activity",
    "mean_jump_height",
    "mean_gait_speed",
    "mean_range_of_motion",
    "drop_y_5",
    "rec_post3",
    "u_drop_3",
    "age",
    "bmi",
]
profile_feature_columns = primary_feature_columns + [
    "mean_sleep_quality",
    "mean_workload",
    "rec_post3",
    "drop_y_5",
    "u_drop_3",
]

em_variable_roles = pd.DataFrame(
    [
        {"variable": "alpha_hat", "role": "primary EM input", "justification": "Athlete-specific workload sensitivity from the preliminary ODE transition."},
        {"variable": "half_life_sessions", "role": "primary EM input", "justification": "Clinically interpretable recovery timescale derived from the ODE transition."},
        {"variable": "r_squared", "role": "primary EM input", "justification": "How coherent the athlete's local fatigue dynamics are under the simple transition model."},
        {"variable": "injury_rate_injured", "role": "primary EM input", "justification": "Empirical injury burden anchor so the profiles are not defined only by latent dynamics."},
        {"variable": "mean_recovery_score", "role": "primary EM input", "justification": "Sustained recovery environment, averaged across the athlete's panel."},
        {"variable": "mean_sleep_quality", "role": "withheld validation", "justification": "External recovery-context check, not used to define the clusters."},
        {"variable": "mean_workload", "role": "withheld validation", "justification": "Checks whether clusters differ in chronic exposure even when not clustered on workload directly."},
        {"variable": "drop_y_5", "role": "withheld validation", "justification": "Direct injury-onset recovery behavior after the latent profiles are estimated."},
        {"variable": "rec_post3", "role": "withheld validation", "justification": "Near-term post-onset recovery score used only to validate the fitted profiles."},
        {"variable": "age / bmi / sport_type / gender", "role": "external annotation", "justification": "Used to check whether the mixture is merely rediscovering demographic or sport strata."},
    ]
)

save_table(em_primary_frame, "table_em_primary_feature_matrix.csv")
save_table(em_onset_features, "table_em_onset_feature_matrix.csv")
save_table(em_variable_roles, "table_em_variable_roles.csv")

display(Markdown("### Athlete-level EM feature matrix"))
display(em_primary_frame.head())
display(Markdown("### Variable roles in the EM analysis"))
em_variable_roles


In [ ]:
# Step 16: repeated-fit EM model selection for the primary feature set.
#
# A single mixture fit is not enough, because Gaussian mixtures can land on local
# optima and can make a more complex solution look attractive just by chance. This
# cell therefore repeats the fit many times for each possible number of profiles and
# records both fit quality and restart stability.
#
# The selection logic is intentionally conservative:
# - low BIC matters,
# - but a solution that changes a lot across reruns is not trusted,
# - and the final choice is the smallest stable solution rather than the flashiest one.
#
# This is the core reason the project ends up with a 3-profile working taxonomy
# rather than simply choosing the single lowest one-run BIC anywhere in the search.
# This section is more careful than the earlier EM attempt:
# we do not trust a single BIC value from a single initialization.

def safe_silhouette(X: np.ndarray, labels: np.ndarray) -> float:
    try:
        return silhouette_score(X, labels)
    except Exception:
        return np.nan


def repeated_gmm_selection(
    X: np.ndarray,
    max_components: int = 6,
    covariance_type: str = "full",
    seeds: list[int] | range = range(15),
    n_init: int = 20,
):
    selection_rows = []
    best_models = {}

    for n_components in range(1, max_components + 1):
        run_rows = []
        label_runs = []
        best_bic = np.inf
        best_model = None
        best_labels = None
        best_probabilities = None
        best_silhouette = np.nan

        for seed in seeds:
            gmm = GaussianMixture(
                n_components=n_components,
                covariance_type=covariance_type,
                n_init=n_init,
                random_state=seed,
            )
            gmm.fit(X)
            probabilities = gmm.predict_proba(X)
            labels = gmm.predict(X)
            bic = gmm.bic(X)
            aic = gmm.aic(X)
            mean_max_prob = probabilities.max(axis=1).mean()
            mean_entropy = -(probabilities * np.log(np.clip(probabilities, 1e-12, 1))).sum(axis=1).mean()
            silhouette_value = np.nan if n_components == 1 else safe_silhouette(X, labels)

            run_rows.append(
                {
                    "seed": seed,
                    "n_components": n_components,
                    "bic": bic,
                    "aic": aic,
                    "mean_max_prob": mean_max_prob,
                    "mean_entropy": mean_entropy,
                    "silhouette": silhouette_value,
                }
            )

            if bic < best_bic:
                best_bic = bic
                best_model = gmm
                best_labels = labels
                best_probabilities = probabilities
                best_silhouette = silhouette_value

            if n_components > 1:
                label_runs.append(labels)

        run_frame = pd.DataFrame(run_rows)
        if n_components > 1:
            ari_scores = []
            for i in range(len(label_runs)):
                for j in range(i + 1, len(label_runs)):
                    ari_scores.append(adjusted_rand_score(label_runs[i], label_runs[j]))
            mean_ari = np.mean(ari_scores)
            min_ari = np.min(ari_scores)
        else:
            mean_ari = np.nan
            min_ari = np.nan

        selection_rows.append(
            {
                "n_components": n_components,
                "best_bic": run_frame["bic"].min(),
                "median_bic": run_frame["bic"].median(),
                "q25_bic": run_frame["bic"].quantile(0.25),
                "q75_bic": run_frame["bic"].quantile(0.75),
                "best_aic": run_frame["aic"].min(),
                "median_max_prob": run_frame["mean_max_prob"].median(),
                "median_entropy": run_frame["mean_entropy"].median(),
                "best_silhouette": best_silhouette,
                "mean_ari": mean_ari,
                "min_ari": min_ari,
            }
        )
        best_models[n_components] = {
            "model": best_model,
            "labels": best_labels,
            "probabilities": best_probabilities,
        }

    return pd.DataFrame(selection_rows), best_models


X_primary = StandardScaler().fit_transform(
    em_primary_frame[primary_feature_columns].fillna(em_primary_frame[primary_feature_columns].median())
)
em_selection_table, em_best_models = repeated_gmm_selection(X_primary, max_components=6, covariance_type="full")

# Covariance sensitivity remains useful, but it is treated as a secondary robustness check.
covariance_rows = []
for covariance_type in ["full", "tied", "diag", "spherical"]:
    for n_components in range(1, 7):
        gmm = GaussianMixture(
            n_components=n_components,
            covariance_type=covariance_type,
            n_init=100,
            random_state=RANDOM_STATE,
        )
        gmm.fit(X_primary)
        covariance_rows.append(
            {
                "covariance_type": covariance_type,
                "n_components": n_components,
                "bic": gmm.bic(X_primary),
                "aic": gmm.aic(X_primary),
            }
        )
em_covariance_sensitivity = pd.DataFrame(covariance_rows)
em_covariance_best = (
    em_covariance_sensitivity.sort_values(["covariance_type", "bic"])
    .drop_duplicates("covariance_type", keep="first")
    .rename(columns={"n_components": "best_bic_components"})
    .reset_index(drop=True)
)

# Choose the main number of profiles from the stable solutions rather than from the single lowest
# one-off BIC value. This avoids over-interpreting unstable local optima.
stable_candidates = em_selection_table.loc[
    (em_selection_table["n_components"] > 1)
    & (em_selection_table["mean_ari"].fillna(0) >= 0.90)
].copy()
if stable_candidates.empty:
    primary_k = int(em_selection_table.sort_values("best_bic").iloc[0]["n_components"])
else:
    primary_k = int(stable_candidates.sort_values(["best_bic", "median_bic"]).iloc[0]["n_components"])

primary_selection_row = em_selection_table.loc[em_selection_table["n_components"] == primary_k].iloc[0]
comparison_k = min(primary_k + 1, int(em_selection_table["n_components"].max()))
unstable_comparison_row = em_selection_table.loc[em_selection_table["n_components"] == comparison_k].iloc[0]

save_table(em_selection_table, "table_em_model_selection_primary.csv")
save_table(em_covariance_sensitivity, "table_em_covariance_sensitivity_primary.csv")
save_table(em_covariance_best, "table_em_covariance_best_primary.csv")

display(Markdown("### Primary EM model selection table"))
display(em_selection_table)
display(Markdown("### Covariance sensitivity for the primary feature set"))
display(em_covariance_best)
display(
    Markdown(
        f'''
**Primary EM model-selection interpretation**

- The primary feature set is `{primary_feature_columns}`.
- Under repeated full-covariance fits, the chosen profile count is **{primary_k}** because it combines
  low BIC with excellent stability (mean ARI = **{primary_selection_row["mean_ari"]:.3f}**).
- A more complex solution can occasionally achieve a lower best-case BIC, but the next higher model
  has much weaker stability (mean ARI = **{unstable_comparison_row["mean_ari"]:.3f}**), which is a warning
  against over-interpreting a local optimum.
- Across covariance structures, the best component count is not identical. That is expected, and it means
  the data support **heterogeneity**, but the exact taxonomy should still be described with some caution.
        '''
    )
)


In [ ]:
# Step 17: fit the selected EM model, label the profiles, and validate them on withheld variables.
#
# Once the number of profiles is fixed, the analysis still has to answer two more
# questions.
#
# First, are the groups interpretable in human terms? That is why the clusters are
# ordered by half-life and then given descriptive labels tied to recovery speed and
# burden rather than arbitrary numbers.
#
# Second, do the groups matter outside the variables that created them? That is why
# the code immediately tests sleep, workload, onset-recovery summaries, sport, and
# gender after fitting the mixture. A latent profile is much more convincing if it
# predicts behavior that was not baked into its definition.
primary_model = em_best_models[primary_k]["model"]
em_primary_assignments = em_primary_frame.copy()
em_primary_assignments["cluster"] = em_best_models[primary_k]["labels"]
em_primary_assignments["cluster_prob"] = em_best_models[primary_k]["probabilities"].max(axis=1)
em_primary_assignments["cluster_entropy"] = -(
    em_best_models[primary_k]["probabilities"]
    * np.log(np.clip(em_best_models[primary_k]["probabilities"], 1e-12, 1))
).sum(axis=1)

cluster_order = (
    em_primary_assignments.groupby("cluster")["half_life_sessions"]
    .mean()
    .sort_values()
    .index
    .tolist()
)
if primary_k == 3:
    cluster_label_map = {
        cluster_order[0]: "Fast recovery / lower burden",
        cluster_order[1]: "Intermediate recovery / higher burden",
        cluster_order[2]: "Slow recovery / high persistence",
    }
else:
    cluster_label_map = {
        cluster_id: f"Recovery profile {i + 1}"
        for i, cluster_id in enumerate(cluster_order)
    }
em_primary_assignments["cluster_label"] = em_primary_assignments["cluster"].map(cluster_label_map)
cluster_label_order = [cluster_label_map[cluster_id] for cluster_id in cluster_order]
palette_values = ["#2a9d8f", "#e76f51", "#577590", "#8d99ae", "#bc6c25", "#6a4c93"]
cluster_palette = {
    label: palette_values[idx]
    for idx, label in enumerate(cluster_label_order)
}

em_cluster_summary = (
    em_primary_assignments.groupby("cluster_label")
    .agg(
        athletes=(athlete_id, "size"),
        mean_alpha_hat=("alpha_hat", "mean"),
        mean_half_life=("half_life_sessions", "mean"),
        mean_r_squared=("r_squared", "mean"),
        mean_injury_rate=("injury_rate_injured", "mean"),
        mean_recovery_score=("mean_recovery_score", "mean"),
        mean_sleep_quality=("mean_sleep_quality", "mean"),
        mean_workload=("mean_workload", "mean"),
        mean_rec_post3=("rec_post3", "mean"),
        mean_drop_y_5=("drop_y_5", "mean"),
        mean_u_drop_3=("u_drop_3", "mean"),
        mean_assignment_prob=("cluster_prob", "mean"),
    )
    .reindex(cluster_label_order)
    .reset_index()
)

validation_rows = []
for variable in validation_feature_columns:
    group_values = [
        group[variable].dropna().values
        for _, group in em_primary_assignments.groupby("cluster_label")
    ]
    anova_stat, anova_p = f_oneway(*group_values)
    kruskal_stat, kruskal_p = kruskal(*group_values)
    grand_mean = em_primary_assignments[variable].mean()
    ss_between = sum(len(values) * (values.mean() - grand_mean) ** 2 for values in group_values)
    ss_total = ((em_primary_assignments[variable] - grand_mean) ** 2).sum()
    eta_sq = ss_between / ss_total if ss_total > 0 else np.nan
    validation_rows.append(
        {
            "variable": variable,
            "anova_pvalue": anova_p,
            "kruskal_pvalue": kruskal_p,
            "eta_squared": eta_sq,
        }
    )
em_validation_summary = (
    pd.DataFrame(validation_rows)
    .sort_values("eta_squared", ascending=False)
    .reset_index(drop=True)
)

categorical_rows = []
for categorical_variable in ["sport_type", "gender"]:
    table = pd.crosstab(em_primary_assignments["cluster_label"], em_primary_assignments[categorical_variable])
    chi2, pvalue, _, _ = chi2_contingency(table)
    n_total = table.values.sum()
    phi2 = chi2 / n_total
    r, c = table.shape
    cramers_v = np.sqrt(phi2 / max(min(r - 1, c - 1), 1))
    categorical_rows.append(
        {
            "variable": categorical_variable,
            "chi_square_pvalue": pvalue,
            "cramers_v": cramers_v,
        }
    )
em_categorical_validation = pd.DataFrame(categorical_rows)

# Cluster-specific injury-onset trajectories provide a direct check that the profiles
# are reflected in real session-level behavior, not only in the summary vectors.
df_with_cluster = df.merge(
    em_primary_assignments[[athlete_id, "cluster_label"]],
    on=athlete_id,
    how="left",
)
onset_window_rows = []
for athlete, athlete_df in df_with_cluster.groupby(athlete_id):
    athlete_df = athlete_df.reset_index(drop=True)
    cluster_label = athlete_df["cluster_label"].iloc[0]
    onset_indices = athlete_df.index[athlete_df["injury_onset"]].tolist()
    for idx in onset_indices:
        for rel_session in range(-2, 4):
            j = idx + rel_session
            if 0 <= j < len(athlete_df):
                onset_window_rows.append(
                    {
                        "cluster_label": cluster_label,
                        "rel_session": rel_session,
                        "u_t": athlete_df.loc[j, "u_t"],
                        "y_t": athlete_df.loc[j, "y_t"],
                        "recovery_score": athlete_df.loc[j, "recovery_score"],
                    }
                )
em_cluster_onset_summary = (
    pd.DataFrame(onset_window_rows)
    .groupby(["cluster_label", "rel_session"])[["u_t", "y_t", "recovery_score"]]
    .mean()
    .reset_index()
)

# PCA is used only for visualization of the primary feature space.
pca = PCA(n_components=2)
pca_scores = pca.fit_transform(X_primary)
em_primary_assignments["pc1"] = pca_scores[:, 0]
em_primary_assignments["pc2"] = pca_scores[:, 1]

profile_display_map = {
    "alpha_hat": "alpha_hat",
    "half_life_sessions": "half_life",
    "r_squared": "transition_R2",
    "injury_rate_injured": "injury_rate",
    "mean_recovery_score": "recovery_score",
    "mean_sleep_quality": "sleep_quality",
    "mean_workload": "mean_workload",
    "rec_post3": "post3_recovery",
    "drop_y_5": "fatigue_drop_5",
    "u_drop_3": "workload_deload_3",
}
profile_heatmap = (
    em_primary_assignments.groupby("cluster_label")[profile_feature_columns]
    .mean()
    .reindex(cluster_label_order)
)
profile_heatmap_z = (
    (profile_heatmap - em_primary_assignments[profile_feature_columns].mean())
    / em_primary_assignments[profile_feature_columns].std(ddof=0)
).rename(columns=profile_display_map)

save_table(em_primary_assignments, "table_em_primary_assignments.csv")
save_table(em_cluster_summary, "table_em_cluster_summary_primary.csv")
save_table(em_validation_summary, "table_em_validation_continuous.csv")
save_table(em_categorical_validation, "table_em_validation_categorical.csv")
save_table(em_cluster_onset_summary, "table_em_cluster_onset_summary.csv")
save_table(profile_heatmap_z.reset_index(), "table_em_profile_heatmap_zscores.csv")

display(Markdown("### Primary EM cluster summary"))
display(em_cluster_summary)
display(Markdown("### Strongest withheld-variable differences across clusters"))
display(em_validation_summary.head(10))
display(Markdown("### Demographic / sport validation"))
display(em_categorical_validation)

strongest_validation = em_validation_summary.iloc[0]
second_validation = em_validation_summary.iloc[1]
third_validation = em_validation_summary.iloc[2]
display(
    Markdown(
        f'''
**Cluster-validation interpretation**

- The largest withheld differences are in **{strongest_validation["variable"]}**,
  **{second_validation["variable"]}**, and **{third_validation["variable"]}**.
  These are not part of the primary clustering vector, so they serve as real external validation.
- In practical terms, the fitted profiles differ not only in latent dynamics, but also in sleep / workload /
  post-onset recovery behavior.
- Sport type and gender are weak explanatory variables here, which is useful: the EM model is not simply
  rediscovering obvious demographic strata.
        '''
    )
)


In [ ]:
# Step 18: generate the new EM-centered figures for the report.
#
# The figures are doing different jobs.
# - Figure 8A explains why the chosen number of profiles is defensible.
# - Figure 8B shows what the latent profiles look like in a lower-dimensional view
#   and in a standardized feature heatmap.
# - Figure 8C checks whether the groups differ on withheld recovery-context variables.
# - Figure 8D shows whether the profiles behave differently around real injury onset.
#
# Together, these plots turn the mixture output from an abstract clustering result
# into something that can be judged in terms of stability, structure, and practical meaning.
selection_plot = em_selection_table.copy()
selection_plot["bic_lower"] = selection_plot["median_bic"] - selection_plot["q25_bic"]
selection_plot["bic_upper"] = selection_plot["q75_bic"] - selection_plot["median_bic"]

# Figure 8A: stable model selection and initialization sensitivity.
fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.3), gridspec_kw={"width_ratios": [1.2, 0.9]})

axes[0].errorbar(
    selection_plot["n_components"],
    selection_plot["median_bic"],
    yerr=[selection_plot["bic_lower"], selection_plot["bic_upper"]],
    fmt="o-",
    color="#1d3557",
    ecolor="#8d99ae",
    elinewidth=1.2,
    capsize=3,
    linewidth=2,
    label="Median BIC (IQR)",
)
axes[0].scatter(
    selection_plot["n_components"],
    selection_plot["best_bic"],
    color="#e76f51",
    s=35,
    label="Best BIC across starts",
    zorder=3,
)
axes[0].axvline(primary_k, color="#2a9d8f", linestyle="--", linewidth=1.5)
axes[0].set_title("Primary EM model selection", pad=6)
axes[0].set_xlabel("Number of profiles")
axes[0].set_ylabel("BIC")
axes[0].set_xticks(selection_plot["n_components"])
axes[0].legend(frameon=False, loc="best")

ari_plot = selection_plot.loc[selection_plot["n_components"] > 1].copy()
ari_colors = ["#2a9d8f" if k == primary_k else "#adb5bd" for k in ari_plot["n_components"]]
axes[1].bar(ari_plot["n_components"], ari_plot["mean_ari"], color=ari_colors, edgecolor="none")
axes[1].set_ylim(0, 1.02)
axes[1].set_title("Repeated-fit stability", pad=6)
axes[1].set_xlabel("Number of profiles")
axes[1].set_ylabel("Mean pairwise ARI")
axes[1].set_xticks(ari_plot["n_components"])
for _, row in ari_plot.iterrows():
    axes[1].annotate(
        f"{row['mean_ari']:.2f}",
        (row["n_components"], row["mean_ari"]),
        ha="center",
        va="bottom",
        fontsize=8,
    )

fig.tight_layout(pad=0.5, w_pad=1.0)
save_figure(fig, "figure_8_em_extension.png")
plt.show()

# Figure 8B: PCA map and standardized cluster profiles.
fig, axes = plt.subplots(1, 2, figsize=(9.1, 3.6), gridspec_kw={"width_ratios": [1.0, 1.15]})

for cluster_label in cluster_label_order:
    cluster_frame = em_primary_assignments.loc[em_primary_assignments["cluster_label"] == cluster_label]
    axes[0].scatter(
        cluster_frame["pc1"],
        cluster_frame["pc2"],
        s=35 + 35 * cluster_frame["cluster_prob"],
        alpha=0.85,
        color=cluster_palette[cluster_label],
        label=cluster_label,
        edgecolor="white",
        linewidth=0.4,
    )
axes[0].set_title("Primary EM profiles in PCA space", pad=6)
axes[0].set_xlabel(f"PC1 ({100 * pca.explained_variance_ratio_[0]:.1f}% var.)")
axes[0].set_ylabel(f"PC2 ({100 * pca.explained_variance_ratio_[1]:.1f}% var.)")
axes[0].legend(frameon=False, loc="best")

sns.heatmap(
    profile_heatmap_z,
    cmap="RdBu_r",
    center=0,
    linewidths=0.4,
    cbar_kws={"shrink": 0.75, "label": "Cluster mean (z-score)"},
    ax=axes[1],
)
axes[1].set_title("Cluster feature profile heatmap", pad=6)
axes[1].set_xlabel("")
axes[1].set_ylabel("")

fig.tight_layout(pad=0.5, w_pad=0.8)
save_figure(fig, "figure_8b_em_robustness.png")
plt.show()

# Figure 8C: withheld-variable validation.
validation_plot_frame = em_cluster_summary.set_index("cluster_label").loc[cluster_label_order].reset_index()
validation_specs = [
    ("mean_sleep_quality", "Sleep quality"),
    ("mean_workload", "Mean workload"),
    ("mean_rec_post3", "Post-onset recovery"),
]
fig, axes = plt.subplots(1, 3, figsize=(9.0, 3.0))
bar_colors = [cluster_palette[label] for label in cluster_label_order]
for ax, (column, title) in zip(axes, validation_specs):
    ax.bar(validation_plot_frame["cluster_label"], validation_plot_frame[column], color=bar_colors, edgecolor="none")
    ax.set_title(title, pad=6)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=25)
axes[0].set_ylabel("Mean value")
axes[1].set_ylabel("")
axes[2].set_ylabel("")
fig.tight_layout(pad=0.5, w_pad=0.8)
save_figure(fig, "figure_8c_em_external_validation.png")
plt.show()

# Figure 8D: cluster-specific injury-onset trajectories.
fig, axes = plt.subplots(1, 3, figsize=(9.2, 3.1), sharex=True)
for ax, variable, title, ylabel in [
    (axes[0], "y_t", "Fatigue around onset", "Fatigue"),
    (axes[1], "recovery_score", "Recovery around onset", "Recovery score"),
    (axes[2], "u_t", "Workload around onset", "Workload"),
]:
    sns.lineplot(
        data=em_cluster_onset_summary,
        x="rel_session",
        y=variable,
        hue="cluster_label",
        hue_order=cluster_label_order,
        palette=cluster_palette,
        linewidth=2,
        marker="o",
        ax=ax,
    )
    ax.set_title(title, pad=6)
    ax.set_xlabel("Sessions rel. onset")
    ax.set_ylabel(ylabel if ax is axes[0] else "")
    if ax is not axes[2]:
        ax.legend_.remove()
axes[2].legend(frameon=False, loc="best")
fig.tight_layout(pad=0.5, w_pad=0.8)
save_figure(fig, "figure_8d_em_onset_profiles.png")
plt.show()

display(
    Markdown(
        f'''
**EM-centered findings for the report**

- The selected mixture is a **{primary_k}-profile** solution on the primary recovery-feature vector.
- That choice is stronger than the earlier EM result because it is based on repeated-fit stability,
  not only on a single BIC value.
- The fitted profiles are externally meaningful: withheld sleep quality, mean workload, and post-onset
  recovery differ across clusters, while sport type and gender do not explain the clustering well.
- In practical terms, the EM analysis now supports **structured latent recovery profiles** rather than a single
  pooled recovery population.
        '''
    )
)


## 4A. Why EM Is Needed Here

The key modeling question is whether one pooled ODE is enough, or whether athletes behave as if they come from
multiple unobserved recovery profiles. That is exactly where the EM layer matters.

The latent variable in this project is the unobserved athlete-level group indicator

$$
z_a \in \{1,\ldots,G\},
$$

which is not directly recorded anywhere in the dataset. It is \emph{latent} because there is no column for
``recovery type,'' ``fast responder,'' or ``slow responder.'' If that hidden structure exists, the pooled ODE
averages over athletes with meaningfully different recovery time scales and workload-response patterns.

In practical terms, we need EM only if all three of the following are true:

1. athlete-level ODE summaries vary materially across athletes;
2. a stable multi-profile mixture fits those summaries better than a one-profile model;
3. the recovered profiles differ on unused external variables rather than only on the variables that created them.

The next cells test those conditions directly.


In [ ]:
# Step 18A: quantify athlete-level heterogeneity and show how the pooled ODE averages over it.
#
# The point of adding EM is not just to say that clusters exist. The stronger claim
# is that a single pooled ODE hides meaningful differences in recovery dynamics.
# This cell tests that directly by refitting the transition model inside each latent
# profile and comparing those profile-specific fits back to the pooled model.
#
# If the pooled model is systematically biased in different directions for different
# clusters, that is strong evidence that one common recovery law is too coarse.
em_primary_assignments = pd.read_csv(OUTPUT_DIR / "table_em_primary_assignments.csv")
em_cluster_summary = pd.read_csv(OUTPUT_DIR / "table_em_cluster_summary_primary.csv")
em_validation_summary = pd.read_csv(OUTPUT_DIR / "table_em_validation_continuous.csv")
em_categorical_validation = pd.read_csv(OUTPUT_DIR / "table_em_validation_categorical.csv")
em_cluster_onset_summary = pd.read_csv(OUTPUT_DIR / "table_em_cluster_onset_summary.csv")
em_selection_table = pd.read_csv(OUTPUT_DIR / "table_em_model_selection_primary.csv")

# Rebuild the smoothed transition panel and merge the selected EM cluster labels.
em_transition_panel = df[
    [athlete_id, "u_t", "y_t_smooth_past", "recovery_score", "sleep_quality", injury_var]
].copy()
em_transition_panel["y_next_smooth_past"] = em_transition_panel.groupby(athlete_id)["y_t_smooth_past"].shift(-1)
em_transition_panel = em_transition_panel.dropna().copy()
em_transition_panel = em_transition_panel.merge(
    em_primary_assignments[[athlete_id, "cluster_label"]],
    on=athlete_id,
    how="left",
)

pooled_transition_model = sm.OLS(
    em_transition_panel["y_next_smooth_past"],
    sm.add_constant(em_transition_panel[["y_t_smooth_past", "u_t"]], has_constant="add"),
).fit()
em_transition_panel["pooled_pred"] = pooled_transition_model.predict(
    sm.add_constant(em_transition_panel[["y_t_smooth_past", "u_t"]], has_constant="add")
)
em_transition_panel["pooled_residual"] = (
    em_transition_panel["y_next_smooth_past"] - em_transition_panel["pooled_pred"]
)

cluster_transition_rows = []
for cluster_label, cluster_frame in em_transition_panel.groupby("cluster_label"):
    cluster_model = sm.OLS(
        cluster_frame["y_next_smooth_past"],
        sm.add_constant(cluster_frame[["y_t_smooth_past", "u_t"]], has_constant="add"),
    ).fit()
    cluster_pred = cluster_model.predict(
        sm.add_constant(cluster_frame[["y_t_smooth_past", "u_t"]], has_constant="add")
    )
    cluster_transition_rows.append(
        {
            "cluster_label": cluster_label,
            "sessions": len(cluster_frame),
            "alpha_hat_cluster": cluster_model.params["u_t"],
            "rho_hat_cluster": cluster_model.params["y_t_smooth_past"],
            "k_hat_cluster": 1 - cluster_model.params["y_t_smooth_past"],
            "adj_r_squared_cluster": cluster_model.rsquared_adj,
            "rmse_cluster_specific_fit": np.sqrt(np.mean((cluster_frame["y_next_smooth_past"] - cluster_pred) ** 2)),
            "rmse_pooled_fit_same_rows": np.sqrt(np.mean((cluster_frame["y_next_smooth_past"] - cluster_frame["pooled_pred"]) ** 2)),
            "mean_pooled_residual": cluster_frame["pooled_residual"].mean(),
            "mae_pooled_residual": cluster_frame["pooled_residual"].abs().mean(),
            "mean_recovery_score": cluster_frame["recovery_score"].mean(),
            "mean_sleep_quality": cluster_frame["sleep_quality"].mean(),
            "injured_session_rate": (cluster_frame[injury_var] == 2).mean(),
        }
    )
em_cluster_transition_summary = pd.DataFrame(cluster_transition_rows)

athlete_primary_dispersion = (
    em_primary_assignments[
        [
            "alpha_hat",
            "half_life_sessions",
            "r_squared",
            "injury_rate_injured",
            "mean_recovery_score",
            "mean_sleep_quality",
            "mean_workload",
        ]
    ]
    .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
    .T.reset_index()
    .rename(columns={"index": "feature"})
)

latent_justification_table = pd.DataFrame(
    [
        {
            "criterion": "Athlete-level recovery-time heterogeneity",
            "evidence": f"Half-life spans {em_primary_assignments['half_life_sessions'].min():.2f} to {em_primary_assignments['half_life_sessions'].max():.2f} sessions.",
            "why_it_matters": "A pooled recovery parameter averages over athletes with materially different time scales.",
        },
        {
            "criterion": "Stable multi-profile EM fit",
            "evidence": (
                f"3-profile solution: best BIC = {em_selection_table.loc[em_selection_table['n_components'] == 3, 'best_bic'].iloc[0]:.1f}, "
                f"mean ARI = {em_selection_table.loc[em_selection_table['n_components'] == 3, 'mean_ari'].iloc[0]:.3f}."
            ),
            "why_it_matters": "This supports structured heterogeneity rather than a single pooled population.",
        },
        {
            "criterion": "Cluster-specific ODE dynamics differ",
            "evidence": (
                f"Cluster-specific k ranges from {em_cluster_transition_summary['k_hat_cluster'].min():.3f} "
                f"to {em_cluster_transition_summary['k_hat_cluster'].max():.3f}."
            ),
            "why_it_matters": "Different latent profiles imply different recovery speeds, not just different burden levels.",
        },
        {
            "criterion": "Pooled ODE is biased by latent profile",
            "evidence": (
                f"Pooled residual means range from {em_cluster_transition_summary['mean_pooled_residual'].min():.3f} "
                f"to {em_cluster_transition_summary['mean_pooled_residual'].max():.3f} across clusters."
            ),
            "why_it_matters": "The single pooled ODE is systematically miscalibrated once latent profiles are ignored.",
        },
        {
            "criterion": "Profiles validate on unused variables",
            "evidence": (
                f"Top withheld effects: rec_post3 eta^2 = {em_validation_summary.loc[em_validation_summary['variable'] == 'rec_post3', 'eta_squared'].iloc[0]:.3f}, "
                f"sleep eta^2 = {em_validation_summary.loc[em_validation_summary['variable'] == 'mean_sleep_quality', 'eta_squared'].iloc[0]:.3f}."
            ),
            "why_it_matters": "The profiles generalize beyond the fitting variables.",
        },
        {
            "criterion": "Profiles are not obvious observed strata",
            "evidence": (
                f"Sport p = {em_categorical_validation.loc[em_categorical_validation['variable'] == 'sport_type', 'chi_square_pvalue'].iloc[0]:.3f}, "
                f"gender p = {em_categorical_validation.loc[em_categorical_validation['variable'] == 'gender', 'chi_square_pvalue'].iloc[0]:.3f}."
            ),
            "why_it_matters": "The group label is latent rather than a re-expression of sport or gender.",
        },
    ]
)

save_table(em_cluster_transition_summary, "table_em_cluster_transition_summary.csv")
save_table(athlete_primary_dispersion, "table_em_primary_feature_dispersion.csv")
save_table(latent_justification_table, "table_em_latent_justification.csv")

display(Markdown("### Athlete-level dispersion in the primary EM features"))
display(athlete_primary_dispersion)
display(Markdown("### Cluster-specific ODE transitions"))
display(em_cluster_transition_summary)
display(Markdown("### Why the latent EM layer is justified"))
latent_justification_table


In [ ]:
# Step 18B: visualize why the latent mixture is needed.
em_cluster_transition_summary = pd.read_csv(OUTPUT_DIR / "table_em_cluster_transition_summary.csv")

cluster_order = em_cluster_transition_summary.sort_values("k_hat_cluster")["cluster_label"].tolist()
latent_palette = {
    label: color
    for label, color in zip(cluster_order, ["#577590", "#e76f51", "#2a9d8f"])
}
summary_plot = (
    em_cluster_transition_summary.set_index("cluster_label")
    .loc[cluster_order]
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.2))

x_positions = np.arange(len(summary_plot))
width = 0.36
bar_colors = [latent_palette[label] for label in summary_plot["cluster_label"]]

axes[0].bar(
    x_positions - width / 2,
    summary_plot["alpha_hat_cluster"],
    width=width,
    color=bar_colors,
    edgecolor="none",
    label=r"$\hat{\alpha}$ by cluster",
)
axes[0].bar(
    x_positions + width / 2,
    summary_plot["k_hat_cluster"],
    width=width,
    color=bar_colors,
    alpha=0.55,
    edgecolor="none",
    label=r"$\hat{k}$ by cluster",
)
axes[0].axhline(pooled_transition_model.params["u_t"], color="#1d3557", linestyle="--", linewidth=1.5)
axes[0].axhline(1 - pooled_transition_model.params["y_t_smooth_past"], color="#6c757d", linestyle=":", linewidth=1.5)
axes[0].set_xticks(x_positions)
axes[0].set_xticklabels(summary_plot["cluster_label"], rotation=20, ha="right")
axes[0].set_title("Cluster-specific ODE dynamics", pad=6)
axes[0].set_ylabel("Coefficient value")
axes[0].legend(frameon=False, loc="best")

axes[1].bar(
    summary_plot["cluster_label"],
    summary_plot["mean_pooled_residual"],
    color=bar_colors,
    edgecolor="none",
)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("Bias of pooled ODE by latent profile", pad=6)
axes[1].set_ylabel("Mean pooled residual")
axes[1].tick_params(axis="x", rotation=20)

fig.tight_layout(pad=0.5, w_pad=0.9)
save_figure(fig, "figure_8e_cluster_specific_ode.png")
plt.show()


In [ ]:
# Step 18C: evaluate the latent-profile argument in plain language.
top_rec = em_validation_summary.loc[em_validation_summary["variable"] == "rec_post3"].iloc[0]
top_sleep = em_validation_summary.loc[em_validation_summary["variable"] == "mean_sleep_quality"].iloc[0]
top_workload = em_validation_summary.loc[em_validation_summary["variable"] == "mean_workload"].iloc[0]

fast_cluster = em_cluster_transition_summary.loc[
    em_cluster_transition_summary["cluster_label"] == "Fast recovery / lower burden"
].iloc[0]
slow_cluster = em_cluster_transition_summary.loc[
    em_cluster_transition_summary["cluster_label"] == "Slow recovery / high persistence"
].iloc[0]

display(
    Markdown(
        f'''
**What do these extra results show?**

- The pooled ODE is averaging across materially different dynamics. In the cluster-specific pooled fits,
  the implied recovery rate is **{fast_cluster['k_hat_cluster']:.3f}** in the fast-recovery cluster but only
  **{slow_cluster['k_hat_cluster']:.3f}** in the slow-recovery cluster.
- The pooled ODE is also systematically biased by latent profile: its mean residual is
  **{fast_cluster['mean_pooled_residual']:.3f}** in the fast-recovery cluster and
  **{slow_cluster['mean_pooled_residual']:.3f}** in the slow-recovery cluster.
- The latent profiles are externally meaningful. The largest withheld validation effects are:
  `rec_post3` with eta^2 = **{top_rec['eta_squared']:.3f}**, `mean_sleep_quality` with eta^2 = **{top_sleep['eta_squared']:.3f}**,
  and `mean_workload` with eta^2 = **{top_workload['eta_squared']:.3f}**.
- Because sport type and gender do not explain the profiles well, the EM grouping is genuinely \
  **latent**: it represents hidden recovery structure rather than a directly observed category.

**Bottom line**

We need EM here because a single average recovery law is too coarse. The ODE identifies athlete-level dynamic
fingerprints, and the EM layer reveals that those fingerprints cluster into stable latent recovery profiles that
also differ on unused recovery and burden variables.
        '''
    )
)


## 5. Conclusion / Discussion


In [ ]:
# Step 19: turn the EM-centered results into a concise report discussion.
#
# The goal here is not to generate polished prose automatically. It is to force the
# notebook to summarize the main statistical claims in one place: what worked, what
# was only preliminary, why the restricted EM stayed the main model, and what still
# limits the interpretation.
top_validation = em_validation_summary.iloc[0]
second_validation = em_validation_summary.iloc[1]
third_validation = em_validation_summary.iloc[2]
best_covariance_components = ", ".join(str(value) for value in em_covariance_best["best_bic_components"].tolist())

display(
    Markdown(
        f'''
The notebook now supports a clearer two-stage story for the project. The ODE is not being used as a final
calibrated mechanistic model yet; instead, it functions as a **preliminary feature-extraction step**. The earlier
sections showed that the smoothed transition model gives the expected workload-accumulation and recovery signs,
while the EM analysis uses those athlete-level summaries to ask the main substantive question: do athletes appear
to follow distinct recovery profiles?

The answer is now stronger than in the previous version, but still nuanced. Using the primary feature set
`{primary_feature_columns}`, the repeated-fit EM analysis selected **{primary_k}** profiles as the best stable
solution under the full-covariance model. The data also support broader heterogeneity under other covariance
assumptions (best component counts = **{best_covariance_components}**), so the conclusion is not that the population
is homogeneous. Instead, the correct reading is that **latent recovery structure is present**, while the exact
high-resolution taxonomy still depends somewhat on modeling choices.

The selected profiles are externally meaningful. The strongest withheld differences occur in
**{top_validation['variable']}**, **{second_validation['variable']}**, and **{third_validation['variable']}**,
which were not used to define the clusters. That matters because it means the mixture is not merely partitioning
noise inside the fitting variables. The cluster-onset plots also show that the profiles differ in real session-level
behavior around injury episodes. By contrast, sport type and gender have weak cluster associations, so the fitted
profiles are not simply demographic relabelings.

For the report, the strongest defensible claim is therefore:

- the ODE is a reasonable first-stage dynamic approximation;
- athlete-level dynamic and recovery features support **multiple recovery profiles** rather than a single pooled population;
- those profiles show meaningful differences in sleep, workload, and post-onset recovery;
- the analysis justifies a later formal EM extension with jointly estimated latent states and recovery groups.
        '''
    )
)


## References

[1] Impellizzeri FM, Ward P, Coutts AJ, Bornn L, McCall A. *Training Load and Injury Part 1: The Devil Is in the Detail-Challenges to Applying the Current Research in the Training Load and Injury Field.* J Orthop Sports Phys Ther. 2020;50(10):574-576.

[2] Maupin D, Schram B, Canetti E, Orr R. *The Relationship Between Acute:Chronic Workload Ratios and Injury Risk in Sports: A Systematic Review.* Open Access J Sports Med. 2020;11:51-75.

[3] Imbach F, Sutton-Charani N, Montmain J, Candau R, Perrey S. *The Use of Fitness-Fatigue Models for Sport Performance Modelling: Conceptual Issues and Contributions from Machine-Learning.* Sports Med Open. 2022;8(1):29.

[4] Dempster AP, Laird NM, Rubin DB. *Maximum Likelihood from Incomplete Data via the EM Algorithm.* Journal of the Royal Statistical Society: Series B. 1977;39(1):1-22.

[5] Bhegam A, et al. *Multimodal Sports Injury Prediction Dataset.* Kaggle dataset documentation bundled with the local archive.
